# OpenFold3: Open-Source Biomolecular Structure Prediction

## CSV Batch Processor for High-Throughput Predictions

Easy-to-use batch processing interface for [OpenFold3](https://github.com/aqlaboratory/openfold3), an open-source reimplementation of AlphaFold3-class biomolecular structure prediction. OpenFold3 delivers accurate predictions for proteins, DNA, RNA, ligands, and their complexes.

### **IMPORTANT: GPU Requirements**
> **OpenFold3 requires a minimum of 32GB VRAM.** Only A100 (40GB/80GB) GPUs are supported.
> T4 (15GB) and L4 (24GB) are **NOT sufficient** and will fail with out-of-memory errors.
> In Google Colab, select **Runtime > Change runtime type > A100 GPU**.

### **Key Features:**
- **Batch Processing**: CSV-based workflow for processing hundreds of predictions efficiently
- **AlphaFold3-Class Accuracy**: High-quality structure prediction for diverse biomolecular systems
- **Comprehensive Support**: Proteins, DNA, RNA, ligands (CCD codes and SMILES), ions
- **Post-Translational Modifications**: Built-in support for 15 common PTMs plus custom modifications
- **Automatic MSA Generation**: Integrated HMMER/kalign for MSA construction
- **Google Drive Integration**: Automatic upload of results to your Drive
- **Open Source**: Fully open weights and code

### **Workflow:**
1. **Prepare CSV**: Create input file with sequences, modifications, and constraints
2. **Configure**: Set prediction parameters (seeds, MSA, precision)
3. **Run Batch**: Automated processing with progress tracking
4. **Download**: Results automatically uploaded to Google Drive

### **CSV Input Format:**
- **Required columns**: `jobname`, `seq1_name`, `seq1_type`, `seq1`
- **Optional columns**: `seq1_copies`, `seq1_mods`
- **Supports**: Up to 10 sequences per job (seq1 through seq10)
- **Sequence types**: protein, dna, rna, ligand (SMILES or CCD code)

### **Supported Modifications:**
- **PTMs**: Phosphorylation (SEP, TPO, PTR), Methylation (MLY, M3L), Acetylation (ALY), and more
- **Ligands**: ATP, GTP, NAD, FAD, SAM, Heme, and 20+ common molecules
- **Ions**: Ca2+, Mg2+, Zn2+, Fe2+/3+, and other metal ions
- **Glycans**: NAG, MAN, GAL, FUC, and other carbohydrate modifications
- **DNA/RNA Mods**: Methylation, oxidation, and other nucleotide modifications
- **Custom**: Upload your own reference CSV with additional modifications

### **Repository:**
- [OpenFold3 GitHub Repository](https://github.com/aqlaboratory/openfold3)

### **Citations:**

**OpenFold3:**

[OpenFold Consortium. OpenFold3: Democratizing Biomolecular Structure Prediction. 2025](https://github.com/aqlaboratory/openfold3)

**If using MSA generation:**

[Eddy SR. Accelerated Profile HMM Searches. *PLoS Computational Biology*, 2011](https://doi.org/10.1371/journal.pcbi.1002195)

---

### **Known Limitations:**
- **GPU requirement**: OpenFold3 requires 32GB+ VRAM. T4 (15GB) and L4 (24GB) are NOT supported. Minimum: A100 (Pro).
- **Preview release**: OpenFold3 v0.3.1 is a preview release and may have stability issues.

### **Quick Start Guide:**

1. **Run Cell 1**: Environment check (verifies A100 GPU)
2. **Run Cell 2**: Install OpenFold3 (takes ~5-10 minutes)
3. **Run Cell 3**: Initialize CSV processor
4. **Run Cell 4**: Upload your CSV file and connect Google Drive
5. **Run Cell 5**: Configure prediction parameters
6. **Run Cell 6**: Execute batch predictions (results auto-uploaded to Drive)

**Example CSV structure:**
```csv
jobname,seq1_name,seq1_type,seq1,seq2_name,seq2_type,seq2
my_protein,proteinA,protein,MSEQVENCE...,ligandATP,ligand,ATP
complex_job,proteinB,protein,MKLLVV...,dna_strand,dna,ATCGATCG
```

For proteins with PTMs: `seq1_mods` column with format `CHAIN:POSITION:CCD` (e.g., `A:10:SEP;A:25:PTR`)

---

**Ready to start? Run Cell 1 below!**

In [ ]:
#@title Cell 1: Environment Check (No Restart Needed)
#@markdown **OpenFold3 is compatible with Colab's default packages.**
#@markdown No kernel restart required. Proceed to Cell 2.
import sys
import os
import subprocess

print("=" * 60)
print(f"{chr(0x1f4e7)} ENVIRONMENT CHECK")
print("=" * 60)

# Check if already installed
ready_marker = "/content/OPENFOLD3_READY"
if os.path.exists(ready_marker):
    print(f"{chr(0x2705)} OpenFold3 already installed. Proceed to Cell 3.")
else:
    result = subprocess.run([sys.executable, "-c", "import numpy; print(f'NumPy: {numpy.__version__}')"], capture_output=True, text=True)
    print(result.stdout.strip())
    result = subprocess.run([sys.executable, "-c", "import torch; print(f'PyTorch: {torch.__version__}')"], capture_output=True, text=True)
    print(result.stdout.strip())
    print(f"\n{chr(0x2705)} Environment compatible. Run Cell 2 to install OpenFold3.")


In [ ]:
#@title Cell 2: Upload CSV and Connect Google Drive
#@markdown Upload your CSV file and connect Google Drive. Then click **Run All Below** for hands-free execution.
%%time

import subprocess
import sys
import os

# Quick-install pydrive2 (not in Colab baseline)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pydrive2"], capture_output=True, text=True)

from google.colab import files

# Configuration options
setup_google_drive = True #@param {type:"boolean"}
#@markdown - Setup Google Drive for automatic result upload

gdrive_folder_name = "OpenFold3_Results" #@param {type:"string"}
#@markdown - Google Drive folder name for batch results

msa_folder = ""  #@param {type:"string"}
#@markdown > **Pre-computed MSAs** (from MSA Colab). Leave empty for online MSA.
#@markdown > Enter a path, zip filename, or Google Drive zip name. The notebook will
#@markdown > resolve it to the `paired/` directory automatically.

print("=" * 60)
print(f"{chr(0x1f4c1)} UPLOAD CSV AND CONNECT GOOGLE DRIVE")
print("=" * 60)

print(f"\n{chr(0x1f4ca)} Upload your input CSV file with job specifications")
print("Required columns: jobname, seq1_name, seq1_type, seq1")
print("Optional: seq1_copies, seq1_mods")
print("Supports up to 10 sequences per job (seq1 through seq10)")

uploaded_csv = files.upload()

if not uploaded_csv:
    raise ValueError("No CSV file uploaded")

csv_filename = list(uploaded_csv.keys())[0]
print(f"\n{chr(0x2705)} Uploaded: {csv_filename}")

# Setup Google Drive
drive = None
if setup_google_drive:
    try:
        from pydrive2.drive import GoogleDrive
        from pydrive2.auth import GoogleAuth
        from google.colab import auth
        from oauth2client.client import GoogleCredentials

        print(f"\n{chr(0x2601)}  Setting up Google Drive...")
        auth.authenticate_user()
        gauth = GoogleAuth()
        gauth.credentials = GoogleCredentials.get_application_default()
        drive = GoogleDrive(gauth)
        print(f"{chr(0x2705)} Google Drive connected")
    except Exception as e:
        print(f"{chr(0x26a0)}  Google Drive setup failed: {e}")
        drive = None

# ============================================================
# RESOLVE PRE-COMPUTED MSA FOLDER
# ============================================================
import os as _os

def resolve_msa_folder(msa_folder_input, _drive=None):
    """Resolve user's msa_folder input to the actual paired/ directory path."""
    if not msa_folder_input or not msa_folder_input.strip():
        return ''

    name = msa_folder_input.strip().rstrip('/')

    if name.startswith('/'):
        base_path = name
    else:
        base_path = f"/content/{name}"

    if _os.path.isdir(base_path):
        paired = _os.path.join(base_path, 'paired')
        if _os.path.isdir(paired) and _os.listdir(paired):
            print(f"   MSA folder resolved: {paired}")
            return paired
        if _os.path.basename(base_path) == 'paired' and _os.listdir(base_path):
            print(f"   MSA folder resolved: {base_path}")
            return base_path

    zip_path = f"{base_path}.zip"
    if not _os.path.isfile(zip_path):
        zip_path = f"/content/{_os.path.basename(name)}.zip"

    if _os.path.isfile(zip_path):
        print(f"   Found local zip: {zip_path}")
        import zipfile as _zf
        extract_to = _os.path.dirname(zip_path) or '/content'
        with _zf.ZipFile(zip_path, 'r') as zf:
            zf.extractall(extract_to)
        print(f"   Unzipped to {extract_to}")
        paired = _os.path.join(base_path, 'paired')
        if _os.path.isdir(paired) and _os.listdir(paired):
            print(f"   MSA folder resolved: {paired}")
            return paired

    if _drive:
        zip_filename = f"{_os.path.basename(name)}.zip"
        print(f"   Searching Google Drive for {zip_filename}...")
        try:
            file_list = _drive.ListFile({
                'q': f"title='{zip_filename}' and trashed=false"
            }).GetList()
            if file_list:
                gdrive_file = file_list[0]
                local_zip = f"/content/{zip_filename}"
                print(f"   Downloading {zip_filename} from Google Drive...")
                gdrive_file.GetContentFile(local_zip)
                print(f"   Downloaded ({_os.path.getsize(local_zip) / 1024 / 1024:.1f} MB)")

                import zipfile as _zf
                with _zf.ZipFile(local_zip, 'r') as zf:
                    zf.extractall('/content')
                print(f"   Unzipped to /content/")

                paired = _os.path.join(base_path, 'paired')
                if _os.path.isdir(paired) and _os.listdir(paired):
                    print(f"   MSA folder resolved: {paired}")
                    return paired
            else:
                print(f"   {zip_filename} not found on Google Drive")
        except Exception as e:
            print(f"   Google Drive download failed: {e}")

    print(f"   WARNING: Could not resolve MSA folder '{msa_folder_input}'")
    return ''

resolved_msa_folder = ''
if msa_folder:
    print(f"\nResolving MSA folder: {msa_folder}")
    resolved_msa_folder = resolve_msa_folder(msa_folder, drive)
    if resolved_msa_folder:
        print(f"   Using pre-computed MSAs from: {resolved_msa_folder}")
        msa_files = [f for f in _os.listdir(resolved_msa_folder) if f.endswith('_paired.a3m')]
        print(f"   Found {len(msa_files)} paired MSA files")

# Store settings (processor and jobs added in Cell 4)
global_settings = {
    'csv_filename': csv_filename,
    'drive': drive,
    'gdrive_folder_name': gdrive_folder_name,
    'msa_folder': resolved_msa_folder,
}

print("\n" + "=" * 60)
print(f"{chr(0x2705)} CSV uploaded, Google Drive {'connected' if drive else 'skipped'}.")
print("Run all cells below for hands-free install + predict.")
print("=" * 60)

In [ ]:
#@title Cell 3: Install OpenFold3
%%time
import subprocess
import sys
import os
import re

def run_cmd(cmd, desc):
    """Execute command with output suppression unless error"""
    print(f"  [{desc}]", end=" ")
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"FAILED")
        print(f"    {result.stderr[:300]}")
        return False
    print("OK")
    return True

def get_gpu_info():
    """Get GPU name and memory"""
    info = {'name': 'Unknown', 'memory_gb': 0.0, 'cuda_version': 'Unknown'}
    try:
        result = subprocess.run(['nvidia-smi'], capture_output=True, text=True, timeout=10)
        if result.returncode == 0:
            match = re.search(r'CUDA Version: (\d+\.\d+)', result.stdout)
            if match:
                info['cuda_version'] = match.group(1)
        result = subprocess.run(
            ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader,nounits'],
            capture_output=True, text=True, timeout=5
        )
        if result.returncode == 0:
            parts = result.stdout.strip().split(',')
            info['name'] = parts[0].strip()
            info['memory_gb'] = float(parts[1].strip()) / 1024.0
    except:
        pass
    return info

# ============================================================
# PREFLIGHT CHECKS
# ============================================================
print("=" * 60)
print("OPENFOLD3 INSTALLATION")
print("=" * 60)

gpu = get_gpu_info()
print(f"\nGPU: {gpu['name']}")
print(f"VRAM: {gpu['memory_gb']:.1f} GB")
print(f"CUDA: {gpu['cuda_version']}")

# CRITICAL: OpenFold3 requires 32GB+ VRAM
if gpu['memory_gb'] < 30.0:
    print("\n" + "!" * 60)
    print("  WARNING: INSUFFICIENT GPU MEMORY")
    print("!" * 60)
    print(f"  Detected: {gpu['memory_gb']:.1f} GB VRAM")
    print(f"  Required: 32 GB minimum (A100 40GB or 80GB)")
    print(f"  OpenFold3 WILL FAIL on T4 (15GB) or L4 (24GB)")
    print(f"  Go to: Runtime > Change runtime type > A100")
    print("!" * 60)
    print("\nInstalling anyway (will check again before running)...")

# Show Colab baseline
print(f"\nColab baseline:")
try:
    import numpy as np
    print(f"  NumPy: {np.__version__}")
except:
    print(f"  NumPy: not found")
try:
    import torch
    print(f"  PyTorch: {torch.__version__}")
    print(f"  CUDA available: {torch.cuda.is_available()}")
except:
    print(f"  PyTorch: not found")

# ============================================================
# INSTALLATION
# ============================================================
# OpenFold3 has NO numpy constraint and NO torch constraint.
# We keep Colab's pre-installed PyTorch 2.9.0 and NumPy 2.0.2.
# This means NO kernel restart is needed!
print("\n" + "=" * 60)
print("INSTALLING DEPENDENCIES")
print("=" * 60)
print("Note: OpenFold3 has no NumPy/PyTorch version pins.")
print("Keeping Colab's pre-installed versions (no restart needed).\n")

# Step 1: System packages
print("[1/6] System packages")
# hmmer for MSA search
run_cmd("apt-get update -qq && apt-get install -y -qq hmmer", "hmmer via apt")

# Step 2: Core Python dependencies that OpenFold3 needs but might not pull
print("\n[2/6] Core Python dependencies")
core_deps = [
    "pytorch-lightning>=2.1",
    "biotite>=1.1.0,<1.3",
    "kalign-python",
    "rdkit",
    "scipy",
    "pandas",
    "PyYAML",
    "tqdm",
    "requests",
    "ml-collections",
    "biopython",
    "lmdb",
    "ijson",
    "func_timeout",
    "memory_profiler",
]
run_cmd(
    f"{sys.executable} -m pip install --quiet {' '.join([repr(d) for d in core_deps])}",
    "Core dependencies"
)

# Step 3: Heavy dependencies
print("\n[3/6] Heavy dependencies (deepspeed, cutlass, cuda-python)")
heavy_deps = [
    "deepspeed",
    "nvidia-cutlass<4",
    "cuda-python<12.9.1",
    "pdbeccdutils",
    "modelcif",
]
run_cmd(
    f"{sys.executable} -m pip install --quiet {' '.join([repr(d) for d in heavy_deps])}",
    "Heavy dependencies"
)

# Step 4: Install OpenFold3
print("\n[4/6] OpenFold3 package")
# Try with cuequivariance first (better performance if CUDA is right)
result = subprocess.run(
    f"{sys.executable} -m pip install --quiet 'openfold3[cuequivariance]==0.4.0'",
    shell=True, capture_output=True, text=True
)
if result.returncode == 0:
    print("  [openfold3[cuequivariance]] OK")
    cueq_pip_installed = True
else:
    print("  [openfold3[cuequivariance]] Failed - trying base install")
    cueq_pip_installed = False
    result2 = subprocess.run(
        f"{sys.executable} -m pip install --quiet 'openfold3==0.4.0'",
        shell=True, capture_output=True, text=True
    )
    if result2.returncode == 0:
        print("  [openfold3] OK (without cuEquivariance)")
    else:
        print(f"  ERROR: Could not install openfold3")
        print(f"  {result2.stderr[:300]}")

# Step 5: Download model weights (direct S3 download — setup_openfold requires conda)
print("\n[5/6] Model weights")
import boto3, botocore

ckpt_dir = os.path.expanduser("~/.openfold3")
os.makedirs(ckpt_dir, exist_ok=True)
ckpt_path = os.path.join(ckpt_dir, "of3-p2-155k.pt")

# Remove old v0.3.1 checkpoint if present (incompatible with v0.4.0)
old_ckpt = os.path.join(ckpt_dir, "of3_ft3_v1.pt")
if os.path.exists(old_ckpt):
    os.remove(old_ckpt)
    print("  Removed old of3_ft3_v1.pt (incompatible with v0.4.0)")

if os.path.exists(ckpt_path) and os.path.getsize(ckpt_path) > 1_000_000_000:
    print(f"  [Model weights] Already present ({os.path.getsize(ckpt_path) / 1e9:.2f} GB)")
else:
    print("  Downloading of3-p2-155k.pt from S3...")
    print("  This may take 2-5 minutes depending on connection speed.")
    try:
        s3 = boto3.client("s3", config=botocore.config.Config(signature_version=botocore.UNSIGNED))
        head = s3.head_object(Bucket="openfold", Key="staging/of3-p2-155k.pt")
        total_size = head['ContentLength']
        print(f"  File size: {total_size / 1e9:.2f} GB")

        downloaded = [0]
        def progress_callback(bytes_transferred):
            downloaded[0] += bytes_transferred
            pct = downloaded[0] / total_size * 100
            if int(pct) % 10 == 0 and abs(downloaded[0] - int(pct/10)*total_size/10) < bytes_transferred:
                print(f"  ... {pct:.0f}%")

        s3.download_file(
            "openfold",
            "staging/of3-p2-155k.pt",
            ckpt_path,
            Callback=progress_callback
        )
        print(f"  [Model weights downloaded] OK ({os.path.getsize(ckpt_path) / 1e9:.2f} GB)")
    except Exception as e:
        print(f"  ERROR: Weight download failed: {e}")
        if os.path.exists(ckpt_path):
            os.remove(ckpt_path)
        print("  Trying fallback: aws s3 cp...")
        result = subprocess.run(
            f"aws s3 cp s3://openfold/staging/of3-p2-155k.pt {ckpt_path} --no-sign-request",
            shell=True, capture_output=True, text=True, timeout=600
        )
        if result.returncode == 0 and os.path.exists(ckpt_path):
            print(f"  [Model weights downloaded via aws CLI] OK")
        else:
            print(f"  ERROR: Both download methods failed. Prediction will fail.")
            print(f"  {result.stderr[:200] if result.stderr else 'No error output'}")

# Write ckpt_root marker so OpenFold3 auto-detection finds the weights
ckpt_root_path = os.path.join(ckpt_dir, "ckpt_root")
with open(ckpt_root_path, "w") as f:
    f.write(ckpt_dir)
print(f"  Checkpoint root: {ckpt_dir}")

# ============================================================
# CUEQUIVARIANCE KERNEL PREFLIGHT TEST
# ============================================================
# NOTE: cuEquivariance on Colab may fail even if pip install succeeds.
# This is similar to the Boltz-2 cuEquivariance issue.
# When it fails, OpenFold3 falls back to standard PyTorch attention
# (slower but works everywhere). If a future Colab update fixes
# cuEquivariance compatibility, this test will pass automatically.

def test_cueq_kernels():
    """Test if cuEquivariance triangle kernels actually work at runtime."""
    print("\n" + "=" * 60)
    print("CUEQUIVARIANCE KERNEL PREFLIGHT TEST")
    print("=" * 60)

    if not cueq_pip_installed:
        print(f"  {chr(0x26a0)}  cuEquivariance not installed (pip failed)")
        return False

    try:
        import torch
        if not torch.cuda.is_available():
            print(f"  {chr(0x26a0)}  CUDA not available")
            return False
    except:
        print(f"  {chr(0x26a0)}  PyTorch/CUDA not available")
        return False

    # Test cuequivariance-torch import
    try:
        import cuequivariance_torch
        print(f"  {chr(0x2705)} cuequivariance-torch: {getattr(cuequivariance_torch, '__version__', 'installed')}")
    except ImportError as e:
        print(f"  {chr(0x274c)} cuequivariance-torch import failed: {e}")
        return False

    # Test cuequivariance-ops-torch import
    try:
        import cuequivariance_ops_torch
        print(f"  {chr(0x2705)} cuequivariance-ops-torch: {getattr(cuequivariance_ops_torch, '__version__', 'installed')}")
    except ImportError as e:
        print(f"  {chr(0x274c)} cuequivariance-ops-torch import failed: {e}")
        return False

    # Test actual kernel import (this is what OpenFold3 uses)
    try:
        import cuequivariance as cue
        print(f"  {chr(0x2705)} cuequivariance: {getattr(cue, '__version__', 'installed')}")
    except ImportError as e:
        print(f"  {chr(0x274c)} cuequivariance import failed: {e}")
        return False

    print(f"  {chr(0x2705)} All cuEquivariance imports successful")
    return True

cueq_available = test_cueq_kernels()

if cueq_available:
    print(f"\n{chr(0x2705)} KERNEL TEST PASSED")
    print("   cuEquivariance kernels available")
    print("   OpenFold3 will use cuEquivariance triangle attention")
else:
    print(f"\n{chr(0x274c)} KERNEL TEST FAILED")
    print("   cuEquivariance kernels NOT available")
    print("   OpenFold3 will use standard PyTorch attention (slower but works)")
    print("   Note: DeepSpeed evoformer attention is always disabled on Colab")
    print("   (requires CUTLASS source tree for JIT compilation)")

# Store result for Cell 6
if 'global_settings' not in globals():
    global_settings = {}
global_settings['cueq_available'] = cueq_available
global_settings['cueq_tested'] = True
print(f"{chr(0x1f527)} Stored: cueq_available = {cueq_available}")


# Step 6: Install ipSAE_batch for interface analysis
print("\n" + "=" * 60)
print("[6/6] ipSAE_batch (interface scoring and visualization)")
run_cmd(
    f"{sys.executable} -m pip install seaborn pycirclize python-igraph plotly",
    "Installing visualization dependencies"
)
if not run_cmd(
    f"{sys.executable} -m pip install git+https://github.com/JKourelis/ipSAE_batch.git",
    "Installing ipSAE_batch"
):
    print("WARNING: ipSAE_batch failed to install. Interface analysis will be unavailable.")

# ============================================================
# VERIFICATION
# ============================================================
print("\n" + "=" * 60)
print("VERIFICATION")
print("=" * 60)

# CLI check
print("\n[Testing OpenFold3 CLI]")
result = subprocess.run("run_openfold predict --help", shell=True,
                      capture_output=True, text=True)
if result.returncode == 0:
    print("  run_openfold predict --help: OK")
else:
    result2 = subprocess.run(f"{sys.executable} -m openfold3 predict --help",
                           shell=True, capture_output=True, text=True)
    if result2.returncode == 0:
        print("  python -m openfold3 predict --help: OK")
    else:
        print("  WARNING: CLI not verified (may still work)")

# kalign check
try:
    import kalign
    print(f"  kalign-python: {getattr(kalign, '__version__', 'installed')}")
except ImportError:
    print("  kalign-python: NOT FOUND (MSA alignment may fail)")

# hmmer check
result = subprocess.run("which hmmsearch 2>/dev/null", shell=True, capture_output=True, text=True)
if result.stdout.strip():
    print(f"  hmmer: OK")
else:
    print("  hmmer: NOT FOUND")

# Test ipSAE_batch
result = subprocess.run(["ipsae-batch", "--help"], capture_output=True, text=True)
if result.returncode == 0:
    print("   ipsae-batch CLI: OK")
else:
    print("   ipsae-batch CLI: NOT AVAILABLE (interface analysis will be skipped)")

# GPU memory check
if gpu['memory_gb'] >= 30.0:
    print(f"\n  GPU: {gpu['name']} ({gpu['memory_gb']:.0f} GB) - SUFFICIENT")
else:
    print(f"\n  GPU: {gpu['name']} ({gpu['memory_gb']:.0f} GB) - INSUFFICIENT (need 32GB+)")

# ============================================================
# ENVIRONMENT & VERSION LOG
# ============================================================
env_lines = []
env_lines.append("\n" + "=" * 60)
env_lines.append("ENVIRONMENT & INSTALLED VERSIONS")
env_lines.append("=" * 60)

# Python version
env_lines.append(f"\nPython: {sys.version.split()[0]}")

# GPU info
gpu_result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    capture_output=True, text=True
)
if gpu_result.returncode == 0:
    env_lines.append(f"GPU: {gpu_result.stdout.strip()}")
env_lines.append(f"CUDA (driver max): {gpu.get('cuda_driver', 'unknown')}")

# PyTorch + CUDA runtime
result = subprocess.run(
    [sys.executable, "-c",
     "import torch; print(f'PyTorch: {torch.__version__}'); "
     "print(f'CUDA available: {torch.cuda.is_available()}'); "
     "print(f'CUDA runtime: {torch.version.cuda}') if torch.cuda.is_available() else None"],
    capture_output=True, text=True
)
if result.returncode == 0:
    for line in result.stdout.strip().split('\n'):
        env_lines.append(f"   {line}")

# All relevant package versions via pip
result = subprocess.run(
    [sys.executable, "-m", "pip", "list", "--format=freeze"],
    capture_output=True, text=True
)
all_packages = result.stdout.strip().split('\n')

# OpenFold3 + core deps
env_lines.append("\nOpenFold3 stack:")
tool_pkgs = [
    'openfold3',
    'numpy',
    'torch',
    'pytorch-lightning',
    'biotite',
    'deepspeed',
    'scipy',
    'rdkit',
    'pandas',
    'lmdb',
    'pdbeccdutils',
    'cuda-python',
    'nvidia-cutlass',
    'cuequivariance',
    'cuequivariance-ops-torch-cu12',
]
for pkg in tool_pkgs:
    pkg_lower = pkg.lower().replace('-', '_')
    for line in all_packages:
        line_lower = line.lower().replace('-', '_')
        if line_lower.startswith(pkg_lower + '=='):
            env_lines.append(f"   {line}")
            break
    else:
        # Try partial match
        for line in all_packages:
            if pkg_lower in line.lower().replace('-', '_'):
                env_lines.append(f"   {line}")
                break

# ipSAE_batch + visualization deps
env_lines.append("\nipSAE_batch stack:")
ipsae_pkgs = [
    'ipsae-batch', 'seaborn', 'pycirclize', 'python-igraph', 'plotly',
    'matplotlib', 'networkx',
]
for pkg in ipsae_pkgs:
    pkg_lower = pkg.lower().replace('-', '_')
    for line in all_packages:
        line_lower = line.lower().replace('-', '_')
        if line_lower.startswith(pkg_lower + '=='):
            env_lines.append(f"   {line}")
            break
    else:
        for line in all_packages:
            if pkg_lower in line.lower().replace('-', '_'):
                env_lines.append(f"   {line}")
                break
        else:
            env_lines.append(f"   {pkg}: not installed")


env_lines.append("\n" + "=" * 60)

# Write environment log to file and print
env_log = "\n".join(str(x) for x in env_lines)
print(env_log)
with open("/content/environment.txt", "w") as _ef:
    _ef.write(env_log + "\n")
print("OPENFOLD3 INSTALLATION COMPLETE")
print("=" * 60)
print("Next: Run Cell 3 to initialize the CSV processor")


In [ ]:
#@title Cell 4: OpenFold3 CSV Processor Setup
import pandas as pd
import os
import re
import json as _json
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any
from io import StringIO

class OpenFold3JobProcessor:
    """Process CSV files for OpenFold3 batch predictions.

    Converts the standard CSV format into OpenFold3's JSON query format:
    {
      "queries": {
        "jobname": {
          "chains": [
            {"chain_ids": ["A"], "molecule_type": "protein", "sequence": "..."},
            {"chain_ids": ["B"], "molecule_type": "ligand", "smiles": "..."}
          ]
        }
      }
    }
    """

    EMBEDDED_REFERENCE = """Type,CCD Code,Name,Target Residue,Molecular Formula,All Atom Count,Heavy Atom Count
PTM,SEP,Phosphoserine,SER,C3H8NO6P,18,11
PTM,TPO,Phosphothreonine,THR,C4H10NO6P,21,12
PTM,PTR,Phosphotyrosine,TYR,C9H12NO6P,28,17
PTM,MLY,N-Methyllysine,LYS,C7H16N2O2,27,11
PTM,ALY,N-Acetyllysine,LYS,C8H16N2O3,29,13
PTM,HYP,Hydroxyproline,PRO,C5H9NO3,18,9
PTM,M3L,N-Trimethyllysine,LYS,C9H20N2O2,33,13
PTM,MLZ,N-Methyllysine,LYS,C7H16N2O2,27,11
PTM,CSD,Cysteine sulfinic acid,CYS,C3H7NO4S,16,9
PTM,CSO,S-Hydroxycysteine,CYS,C3H7NO3S,15,8
PTM,CGU,Gamma-carboxyglutamic acid,GLU,C5H7NO6,21,12
PTM,FME,N-Formylmethionine,MET,C6H11NO3S,22,11
PTM,NEP,N-(phosphonoethyl)isoleucine,ILE,C8H18NO5P,32,15
PTM,HIC,4-Methyl-histidine,HIS,C7H11N3O2,23,12
PTM,CAS,S-(dimethylarsenic)cysteine,CYS,C5H11AsNO2S,20,9
Ligand,ATP,Adenosine triphosphate,NA,C10H16N5O13P3,47,31
Ligand,ADP,Adenosine diphosphate,NA,C10H15N5O10P2,42,27
Ligand,AMP,Adenosine monophosphate,NA,C10H14N5O7P,37,23
Ligand,GTP,Guanosine triphosphate,NA,C10H16N5O14P3,48,32
Ligand,GDP,Guanosine diphosphate,NA,C10H15N5O11P2,43,28
Ligand,GMP,Guanosine monophosphate,NA,C10H14N5O8P,38,24
Ligand,CTP,Cytidine triphosphate,NA,C9H16N3O14P3,45,29
Ligand,CDP,Cytidine diphosphate,NA,C9H15N3O11P2,40,25
Ligand,UTP,Uridine triphosphate,NA,C9H15N2O15P3,44,29
Ligand,UDP,Uridine diphosphate,NA,C9H14N2O12P2,39,25
Ligand,NAD,Nicotinamide adenine dinucleotide,NA,C21H27N7O14P2,71,44
Ligand,NAP,NADP,NA,C21H28N7O17P3,86,55
Ligand,FAD,Flavin adenine dinucleotide,NA,C27H33N9O15P2,91,53
Ligand,FMN,Flavin mononucleotide,NA,C17H21N4O9P,52,31
Ligand,HEM,Heme,NA,C34H32FeN4O4,75,43
Ligand,SAH,S-Adenosyl-L-homocysteine,NA,C14H20N6O5S,46,26
Ligand,SAM,S-Adenosyl-L-methionine,NA,C15H22N6O5S,49,27
Ligand,COA,Coenzyme A,NA,C21H36N7O16P3S,90,57
Ligand,ACO,Acetyl coenzyme A,NA,C23H38N7O17P3S,99,61
Ligand,PLP,Pyridoxal-5-phosphate,NA,C8H10NO6P,25,16
Ligand,TPP,Thiamine diphosphate,NA,C12H19N4O7P2S,45,25
Ligand,BTN,Biotin,NA,C10H16N2O3S,32,16
Ligand,MTA,5-Methylthioadenosine,NA,C11H15N5O3S,35,20
Ligand,THM,Thiamine,NA,C12H17ClN4OS,38,18
Ion,MG,Magnesium ion,NA,Mg,1,1
Ion,ZN,Zinc ion,NA,Zn,1,1
Ion,CA,Calcium ion,NA,Ca,1,1
Ion,FE,Iron ion,NA,Fe,1,1
Ion,MN,Manganese ion,NA,Mn,1,1
Ion,CU,Copper ion,NA,Cu,1,1
Ion,CO,Cobalt ion,NA,Co,1,1
Ion,NI,Nickel ion,NA,Ni,1,1
Ion,K,Potassium ion,NA,K,1,1
Ion,NA,Sodium ion,NA,Na,1,1
Ion,CL,Chloride ion,NA,Cl,1,1
Glycan,NAG,N-Acetyl-D-glucosamine,NA,C8H15NO6,30,15
Glycan,MAN,D-Mannose,NA,C6H12O6,24,12
Glycan,FUC,L-Fucose,NA,C6H12O5,23,11
Glycan,GAL,D-Galactose,NA,C6H12O6,24,12
Glycan,SIA,N-Acetylneuraminic acid,NA,C11H19NO9,40,21
Glycan,GLC,D-Glucose,NA,C6H12O6,24,12
Glycan,BMA,beta-D-Mannose,NA,C6H12O6,24,12
Glycan,NDG,N-Acetyl-D-glucosamine,NA,C8H15NO6,30,15
Glycan,A2G,N-Acetyl-D-galactosamine,NA,C8H15NO6,30,15
Glycan,FUL,L-Fucose,NA,C6H12O5,23,11
DNA_Mod,5MC,5-Methylcytosine,DC,C10H15N3O7P,36,21
DNA_Mod,6MA,N6-Methyladenine,DA,C11H16N5O6P,39,23
DNA_Mod,5HMC,5-Hydroxymethylcytosine,DC,C10H15N3O8P,37,22
DNA_Mod,8OG,8-Oxoguanine,DG,C10H13N5O8P,37,24
DNA_Mod,M2G,N2-Methylguanine,DG,C11H16N5O7P,40,24
DNA_Mod,4MC,N4-Methylcytosine,DC,C10H15N3O7P,36,21
DNA_Mod,1MA,1-Methyladenine,DA,C11H16N5O6P,39,23
DNA_Mod,3MA,3-Methyladenine,DA,C11H16N5O6P,39,23
RNA_Mod,PSU,Pseudouridine,U,C9H12N2O9P,33,21
RNA_Mod,1MA,1-Methyladenosine,A,C11H15N5O7P,39,24
RNA_Mod,7MG,7-Methylguanosine,G,C11H15N5O8P,40,25
RNA_Mod,5MC,5-Methylcytidine,C,C10H15N3O8P,37,22
RNA_Mod,2MA,2-Methyladenosine,A,C11H15N5O7P,39,24
RNA_Mod,M2G,N2-Methylguanosine,G,C11H15N5O8P,40,25
RNA_Mod,M7G,7-Methylguanosine,G,C11H15N5O8P,40,25
RNA_Mod,M1A,1-Methyladenosine,A,C11H15N5O7P,39,24
RNA_Mod,OMC,2'-O-Methylcytidine,C,C10H15N3O8P,37,22
RNA_Mod,OMG,2'-O-Methylguanosine,G,C11H15N5O8P,40,25"""

    # SMILES lookup for common ligands (for OpenFold3 ligand input)
    LIGAND_SMILES = {
        "ATP": "c1nc(c2c(n1)n(cn2)[C@@H]3[C@@H]([C@@H]([C@H](O3)COP(=O)(O)OP(=O)(O)OP(=O)(O)O)O)O)N",
        "ADP": "c1nc(c2c(n1)n(cn2)[C@@H]3[C@@H]([C@@H]([C@H](O3)COP(=O)(O)OP(=O)(O)O)O)O)N",
        "AMP": "c1nc(c2c(n1)n(cn2)[C@@H]3[C@@H]([C@@H]([C@H](O3)COP(=O)(O)O)O)O)N",
        "GTP": "c1[nH]c2c(n1)c(=O)[nH]c(n2)[C@@H]3[C@@H]([C@@H]([C@H](O3)COP(=O)(O)OP(=O)(O)OP(=O)(O)O)O)O",
        "GDP": "c1[nH]c2c(n1)c(=O)[nH]c(n2)[C@@H]3[C@@H]([C@@H]([C@H](O3)COP(=O)(O)OP(=O)(O)O)O)O",
        "GMP": "c1[nH]c2c(n1)c(=O)[nH]c(n2)[C@@H]3[C@@H]([C@@H]([C@H](O3)COP(=O)(O)O)O)O",
        "CTP": "c1cn(c(=O)nc1N)[C@@H]2[C@@H]([C@@H]([C@H](O2)COP(=O)(O)OP(=O)(O)OP(=O)(O)O)O)O",
        "CDP": "c1cn(c(=O)nc1N)[C@@H]2[C@@H]([C@@H]([C@H](O2)COP(=O)(O)OP(=O)(O)O)O)O",
        "UTP": "c1cn(c(=O)[nH]c1=O)[C@@H]2[C@@H]([C@@H]([C@H](O2)COP(=O)(O)OP(=O)(O)OP(=O)(O)O)O)O",
        "UDP": "c1cn(c(=O)[nH]c1=O)[C@@H]2[C@@H]([C@@H]([C@H](O2)COP(=O)(O)OP(=O)(O)O)O)O",
        "NAD": "c1cc(c[n+](c1)[C@@H]2[C@@H]([C@@H]([C@H](O2)COP(=O)([O-])OP(=O)(O)OC[C@@H]3[C@H]([C@H]([C@@H](O3)n4cnc5c4ncnc5N)O)O)O)O)C(=O)N",
        "NAP": "c1cc(c[n+](c1)[C@@H]2[C@@H]([C@@H]([C@H](O2)COP(=O)([O-])OP(=O)(O)OC[C@@H]3[C@H]([C@H]([C@@H](O3)n4cnc5c4ncnc5N)OP(=O)(O)O)O)O)O)C(=O)N",
        "FAD": "Cc1cc2c(cc1C)N(C3=NC(=O)NC(=O)C3=N2)C[C@@H](C[C@@H](CO)O)OP(=O)(O)OP(=O)(O)OC[C@@H]4[C@H]([C@H]([C@@H](O4)n5cnc6c5ncnc6N)O)O",
        "FMN": "Cc1cc2c(cc1C)n(c3c(n2)c(=O)[nH]c(=O)n3)C[C@@H](C[C@@H](COP(=O)(O)O)O)O",
        "HEM": "C=CC1=C(C2=CC3=C(C(=C(N3)C=C4C(=C(C(=N4)C=C5C(=C(C(=N5)C=C1N2)C)C=C)C)CC(=O)O)CC(=O)O)C)C.[Fe]",
        "SAH": "c1nc(c2c(n1)n(cn2)[C@@H]3[C@@H]([C@@H]([C@H](O3)CSCC[C@@H](C(=O)O)N)O)O)N",
        "SAM": "c1nc(c2c(n1)n(cn2)[C@@H]3[C@@H]([C@@H]([C@H](O3)CS(CC[C@@H](C(=O)O)N)C)O)O)N",
        "COA": "CC(C)(COP(=O)(O)OP(=O)(O)OC[C@@H]1[C@H]([C@H]([C@@H](O1)n2cnc3c2ncnc3N)O)OP(=O)(O)O)C(C(=O)NCCC(=O)NCCS)O",
        "ACO": "CC(=O)SCCNC(=O)CCNC(=O)C(C(C)(C)COP(=O)(O)OP(=O)(O)OC[C@@H]1[C@H]([C@H]([C@@H](O1)n2cnc3c2ncnc3N)O)OP(=O)(O)O)O",
        "PLP": "Cc1c(c(c(cn1)COP(=O)(O)O)C=O)O",
        "TPP": "Cc1c(sc(=[n+]1Cc2cnc(nc2N)C)CCOP(=O)(O)OP(=O)(O)O)C",
        "BTN": "C1[C@H]2[C@@H](SC1CCCCC(=O)O)NC(=O)N2",
        "MTA": "c1nc(c2c(n1)n(cn2)[C@@H]3[C@@H]([C@@H]([C@H](O3)CSC)O)O)N",
        "THM": "Cc1c(sc(=[n+]1Cc2cnc(nc2N)C)CCO)C",
    }

    # SMILES for ions
    ION_SMILES = {
        "MG": "[Mg+2]",
        "ZN": "[Zn+2]",
        "CA": "[Ca+2]",
        "FE": "[Fe+2]",
        "MN": "[Mn+2]",
        "CU": "[Cu+2]",
        "CO": "[Co+2]",
        "NI": "[Ni+2]",
        "K": "[K+]",
        "NA": "[Na+]",
        "CL": "[Cl-]",
    }

    def __init__(self, reference_csv: Optional[str] = None):
        """Initialize processor with reference data"""
        if reference_csv:
            self.reference_data = pd.read_csv(StringIO(reference_csv))
        else:
            self.reference_data = pd.read_csv(StringIO(self.EMBEDDED_REFERENCE))

        self.ptm_lookup = self._create_lookup('PTM')
        self.ligand_lookup = self._create_lookup('Ligand')
        self.ion_lookup = self._create_lookup('Ion')
        self.glycan_lookup = self._create_lookup('Glycan')
        self.dna_mod_lookup = self._create_lookup('DNA_Mod')
        self.rna_mod_lookup = self._create_lookup('RNA_Mod')

        self.aa_3to1 = {
            'ALA': 'A', 'CYS': 'C', 'ASP': 'D', 'GLU': 'E', 'PHE': 'F',
            'GLY': 'G', 'HIS': 'H', 'ILE': 'I', 'LYS': 'K', 'LEU': 'L',
            'MET': 'M', 'ASN': 'N', 'PRO': 'P', 'GLN': 'Q', 'ARG': 'R',
            'SER': 'S', 'THR': 'T', 'VAL': 'V', 'TRP': 'W', 'TYR': 'Y'
        }

        self.amino_acids = set('ACDEFGHIKLMNPQRSTVWY')

    def _create_lookup(self, type_name: str) -> Dict[str, Dict[str, Any]]:
        """Create lookup dictionary for a specific type"""
        type_data = self.reference_data[self.reference_data['Type'] == type_name]
        lookup = {}
        for _, row in type_data.iterrows():
            if pd.notna(row['CCD Code']):
                lookup[row['CCD Code']] = {
                    'name': row['Name'],
                    'target_residue': row['Target Residue'] if pd.notna(row['Target Residue']) else 'NA'
                }
        return lookup

    def _validate_sequence_characters(self, sequence: str, seq_type: str) -> List[str]:
        """Validate sequence contains only allowed characters"""
        errors = []
        sequence = sequence.upper()

        if seq_type.lower() == 'protein':
            invalid_chars = []
            for i, char in enumerate(sequence, 1):
                if char.isalpha() and char not in self.amino_acids:
                    invalid_chars.append(f"{char}@{i}")
                elif not char.isalpha() and not char.isspace():
                    invalid_chars.append(f"{char}@{i}")
            if invalid_chars:
                errors.append(f"Invalid amino acids: {', '.join(invalid_chars[:10])}" +
                            ("..." if len(invalid_chars) > 10 else ""))

        elif seq_type.lower() == 'dna':
            valid_bases = set('ATCG')
            invalid_chars = []
            for i, char in enumerate(sequence, 1):
                if char.isalpha() and char not in valid_bases:
                    invalid_chars.append(f"{char}@{i}")
                elif not char.isalpha() and not char.isspace():
                    invalid_chars.append(f"{char}@{i}")
            if invalid_chars:
                errors.append(f"Invalid DNA bases: {', '.join(invalid_chars[:10])}" +
                            ("..." if len(invalid_chars) > 10 else ""))

        elif seq_type.lower() == 'rna':
            valid_bases = set('AUCG')
            invalid_chars = []
            for i, char in enumerate(sequence, 1):
                if char.isalpha() and char not in valid_bases:
                    invalid_chars.append(f"{char}@{i}")
                elif not char.isalpha() and not char.isspace():
                    invalid_chars.append(f"{char}@{i}")
            if invalid_chars:
                errors.append(f"Invalid RNA bases: {', '.join(invalid_chars[:10])}" +
                            ("..." if len(invalid_chars) > 10 else ""))

        return errors

    def _is_smiles(self, ligand_string: str) -> bool:
        """Check if string is likely a SMILES representation"""
        if len(ligand_string) < 3:
            return False
        smiles_chars = set('[]()=#@+-\\/CNOPSFClBrI0123456789')
        return any(char in smiles_chars for char in ligand_string)

    def _get_ligand_smiles(self, ccd_code: str) -> Optional[str]:
        """Look up SMILES for a CCD code from embedded data"""
        if ccd_code in self.LIGAND_SMILES:
            return self.LIGAND_SMILES[ccd_code]
        if ccd_code in self.ION_SMILES:
            return self.ION_SMILES[ccd_code]
        return None

    def _remap_modification_chains(self, mod_string: str, name_mapping: Dict[str, str]) -> str:
        """Remap modification chain IDs from user names to A, B, C... format"""
        if pd.isna(mod_string) or str(mod_string).strip() == '':
            return mod_string

        mod_string = str(mod_string).strip()

        for old_name, new_chain_id in name_mapping.items():
            mod_string = mod_string.replace(f"{old_name}:", f"{new_chain_id}:")

        return mod_string

    def _parse_modifications(self, mod_string: str, sequence: str, chain_id: str, seq_type: str) -> Tuple[List[Dict], List[str]]:
        """Parse modifications in format: CHAIN:POSITION:CCD_CODE"""
        errors = []
        mods = []

        if pd.isna(mod_string) or str(mod_string).strip() == '':
            return mods, errors

        mod_string = str(mod_string).strip()
        mod_entries = [e.strip() for e in mod_string.split(';') if e.strip()]

        if seq_type.lower() == 'protein':
            mod_lookups = [self.ptm_lookup, self.glycan_lookup]
            mod_types = ['PTM', 'Glycan']
        elif seq_type.lower() == 'dna':
            mod_lookups = [self.dna_mod_lookup]
            mod_types = ['DNA Mod']
        elif seq_type.lower() == 'rna':
            mod_lookups = [self.rna_mod_lookup]
            mod_types = ['RNA Mod']
        else:
            return mods, errors

        for entry in mod_entries:
            parts = entry.split(':')
            if len(parts) != 3:
                errors.append(f"Invalid modification format: '{entry}'. Use CHAIN:POSITION:CCD_CODE")
                continue

            mod_chain, pos_str, ccd_code = parts
            mod_chain = mod_chain.strip()
            ccd_code = ccd_code.strip()

            if mod_chain != chain_id:
                continue

            try:
                position = int(pos_str.strip())

                found = False
                for lookup, mod_type in zip(mod_lookups, mod_types):
                    if ccd_code in lookup:
                        found = True

                        if position < 1 or position > len(sequence):
                            errors.append(f"{mod_type} position {position} out of range (sequence length: {len(sequence)})")
                            continue

                        target_residue = lookup[ccd_code]['target_residue']
                        actual_residue = sequence[position - 1].upper()

                        if target_residue != 'NA':
                            if mod_type == 'Glycan':
                                if actual_residue not in ['N', 'T', 'S']:
                                    errors.append(f"Glycan {ccd_code} requires N/T/S but found {actual_residue} at position {position}")
                                    continue
                            elif mod_type == 'PTM':
                                target_1letter = self.aa_3to1.get(target_residue.upper())
                                if target_1letter and actual_residue != target_1letter:
                                    errors.append(f"PTM {ccd_code} targets {target_residue}({target_1letter}) but found {actual_residue} at position {position}")
                                    continue
                            else:
                                if len(target_residue) > 1 and target_residue.startswith('D'):
                                    target_residue = target_residue[1]
                                if actual_residue != target_residue:
                                    errors.append(f"{mod_type} {ccd_code} targets {target_residue} but found {actual_residue} at position {position}")
                                    continue

                        mods.append({
                            'chain_id': mod_chain,
                            'position': position,
                            'ccd': ccd_code
                        })
                        break

                if not found:
                    errors.append(f"Unknown modification code: '{ccd_code}'")

            except ValueError:
                errors.append(f"Invalid position in modification: '{entry}'")

        return mods, errors

    def _sanitize_jobname(self, jobname: str) -> Tuple[str, List[str]]:
        """Sanitize jobname for filesystem compatibility"""
        errors = []

        if pd.isna(jobname) or str(jobname).strip() == '':
            errors.append("Missing jobname")
            return '', errors

        original = str(jobname)
        sanitized = original.lower()
        sanitized = sanitized.replace('-', '_')
        sanitized = re.sub(r'[^a-z0-9_]', '', sanitized)

        if len(sanitized) > 128:
            sanitized = sanitized[:128]
            errors.append(f"Jobname truncated to 128 characters")

        if not sanitized.strip():
            errors.append(f"Jobname '{original}' became empty after sanitization")
            return '', errors

        return sanitized, errors

    def _generate_query_json(self, job: Dict) -> Dict:
        """Generate OpenFold3 JSON query format for a single job.

        Returns a dict suitable for the 'queries' section:
        {
          "chains": [
            {"chain_ids": ["A"], "molecule_type": "protein", "sequence": "..."},
            {"chain_ids": ["B"], "molecule_type": "ligand", "smiles": "..."},
          ]
        }
        """
        chains = []

        # Group sequences by identical content for multi-chain entries
        # OpenFold3 supports multiple chain_ids per entry for copies
        processed = set()

        for i, seq in enumerate(job['sequences']):
            if i in processed:
                continue

            seq_type = seq.get('type', '').lower()
            chain_id = seq.get('id', '')

            if seq_type in ['protein', 'dna', 'rna']:
                # Find all copies (same sequence and type)
                matching_ids = [chain_id]
                for j, other in enumerate(job['sequences']):
                    if j <= i or j in processed:
                        continue
                    if (other.get('type', '').lower() == seq_type and
                        other.get('sequence', '') == seq.get('sequence', '')):
                        matching_ids.append(other.get('id', ''))
                        processed.add(j)

                # Map molecule_type for OpenFold3
                mol_type = seq_type  # protein, dna, rna map directly

                chain_entry = {
                    "chain_ids": matching_ids,
                    "molecule_type": mol_type,
                    "sequence": seq.get('sequence', '')
                }

                # Add modifications if present
                all_mods = seq.get('modifications')
                if all_mods:
                    chain_entry["modifications"] = [
                        {"position": m['position'], "ccd_code": m['ccd']}
                        for m in all_mods
                    ]

                chains.append(chain_entry)

            elif seq_type == 'ligand':
                # Find all copies (same ligand)
                matching_ids = [chain_id]
                for j, other in enumerate(job['sequences']):
                    if j <= i or j in processed:
                        continue
                    if other.get('type', '').lower() == 'ligand':
                        same = False
                        if 'smiles' in seq and 'smiles' in other:
                            same = seq['smiles'] == other['smiles']
                        elif 'ccd' in seq and 'ccd' in other:
                            same = seq['ccd'] == other['ccd']
                        if same:
                            matching_ids.append(other.get('id', ''))
                            processed.add(j)

                chain_entry = {
                    "chain_ids": matching_ids,
                    "molecule_type": "ligand"
                }

                if 'smiles' in seq:
                    chain_entry["smiles"] = seq['smiles']
                elif 'ccd' in seq:
                    ccd_code = seq['ccd']
                    # Convert CCD to SMILES for OpenFold3
                    smiles = self._get_ligand_smiles(ccd_code)
                    if smiles:
                        chain_entry["smiles"] = smiles
                    else:
                        # Use CCD code directly (OpenFold3 may support it)
                        chain_entry["ccd_code"] = ccd_code

                chains.append(chain_entry)

            processed.add(i)

        return {"chains": chains}

    def _process_job(self, row: pd.Series) -> Tuple[Optional[Dict], List[str]]:
        """Process a single job row from CSV"""
        errors = []
        all_sequences = []

        jobname, jobname_errors = self._sanitize_jobname(row.get('jobname', ''))
        errors.extend(jobname_errors)

        if not jobname:
            return None, errors

        chain_id_counter = 0
        name_to_chain_mapping = {}

        for i in range(1, 11):
            name_col = f'seq{i}_name'
            type_col = f'seq{i}_type'
            copies_col = f'seq{i}_copies'
            seq_col = f'seq{i}'
            mods_col = f'seq{i}_mods'

            if name_col not in row or type_col not in row or seq_col not in row:
                continue

            user_name = row.get(name_col, '')
            seq_type = row.get(type_col, '')
            seq_type = str(seq_type).strip().lower()
            copies = row.get(copies_col, 1)
            sequence = row.get(seq_col, '')
            mods = row.get(mods_col, '')

            if pd.isna(sequence) or str(sequence).strip() == '':
                continue

            sequence = str(sequence).strip()
            copies = int(copies) if pd.notna(copies) and str(copies).strip() != '' else 1
            user_name = str(user_name).strip() if pd.notna(user_name) else ''

            chain_ids = []
            for copy_num in range(copies):
                chain_id = chr(ord('A') + chain_id_counter)
                chain_ids.append(chain_id)

                if user_name and copy_num == 0:
                    name_to_chain_mapping[user_name] = chain_id

                chain_id_counter += 1

            if seq_type.lower() in ['protein', 'dna', 'rna']:
                char_errors = self._validate_sequence_characters(sequence, seq_type)
                errors.extend(char_errors)

                for idx, chain_id in enumerate(chain_ids):
                    remapped_mods = self._remap_modification_chains(mods, name_to_chain_mapping)
                    mods_list, mod_errors = self._parse_modifications(remapped_mods, sequence, chain_id, seq_type)
                    errors.extend(mod_errors)

                    seq_dict = {
                        'type': seq_type.lower(),
                        'id': chain_id,
                        'name': user_name,
                        'sequence': sequence,
                        'modifications': mods_list if mods_list else None
                    }
                    all_sequences.append(seq_dict)

            elif seq_type.lower() == 'ligand':
                ligand_string = sequence.strip()

                # Check CCD lookup FIRST (before SMILES detection)
                # _is_smiles has false positives for CCD codes containing P,S,N,C,O,F,I
                looked_up_smiles = None
                is_known_ccd = False
                if ligand_string in self.ligand_lookup:
                    looked_up_smiles = self.ligand_lookup[ligand_string].get('smiles')
                    is_known_ccd = True
                elif ligand_string in self.ion_lookup:
                    looked_up_smiles = self.ion_lookup[ligand_string].get('smiles')
                    is_known_ccd = True

                if not is_known_ccd and not self._is_smiles(ligand_string):
                    errors.append(f"Unknown ligand/ion: '{ligand_string}'")

                for chain_id in chain_ids:
                    if looked_up_smiles:
                        seq_dict = {
                            'type': 'ligand',
                            'id': chain_id,
                            'smiles': looked_up_smiles
                        }
                    elif is_known_ccd:
                        seq_dict = {
                            'type': 'ligand',
                            'id': chain_id,
                            'ccd': ligand_string
                        }
                    else:
                        seq_dict = {
                            'type': 'ligand',
                            'id': chain_id,
                            'smiles': ligand_string
                        }
                    all_sequences.append(seq_dict)

            else:
                errors.append(f"Unsupported sequence type: {seq_type}")

        if not all_sequences:
            errors.append("No valid sequences found")
            return None, errors

        has_modifications = any(seq.get('modifications') for seq in all_sequences)

        job = {
            'name': jobname,
            'sequences': all_sequences,
            'has_modifications': has_modifications,
        }

        return job, errors

    def process_csv(self, csv_path: str) -> Tuple[List[Dict], pd.DataFrame]:
        """Process CSV file and return jobs list and summary DataFrame"""
        df = pd.read_csv(csv_path)

        jobs = []
        summary_rows = []

        for idx, row in df.iterrows():
            job, errors = self._process_job(row)

            if job:
                jobs.append(job)
                summary_rows.append({
                    'jobname': job['name'],
                    'sequences': len(job['sequences']),
                    'has_modifications': job['has_modifications'],
                    'status': 'ERROR' if errors else 'OK',
                    'errors': '; '.join(errors) if errors else ''
                })
            else:
                summary_rows.append({
                    'jobname': f"Row {idx+1}",
                    'sequences': 0,
                    'has_modifications': False,
                    'status': 'FAILED',
                    'errors': '; '.join(errors)
                })

        summary_df = pd.DataFrame(summary_rows)
        return jobs, summary_df

print("Initializing OpenFold3 CSV Processor...")
openfold3_processor = OpenFold3JobProcessor()

print("Using embedded reference data: 79 entries")
print("Processor ready")
print("Reference data includes:")
print(f"   - 15 PTM types")
print(f"   - 24 ligand types (with SMILES lookup)")
print(f"   - 11 ion types (with SMILES lookup)")
print(f"   - 10 glycan types")
print(f"   - 8 DNA modification types")
print(f"   - 10 RNA modification types")
print(f"\nSMILES lookup available for {len(OpenFold3JobProcessor.LIGAND_SMILES)} ligands + {len(OpenFold3JobProcessor.ION_SMILES)} ions")
print("\nNote: Chain IDs are assigned as A, B, C, D... sequentially")
print("   OpenFold3 uses JSON query format (auto-generated from CSV)")
print("   To use custom reference: upload file in Cell 3")

# ============================================================
# PROCESS UPLOADED CSV
# ============================================================
import pandas as pd

if 'global_settings' not in globals() or 'csv_filename' not in global_settings:
    print("Error: Please run Cell 2 (Upload CSV) first")
    raise ValueError("No CSV uploaded")

csv_filename = global_settings['csv_filename']

# Initialize processor
custom_ref_file = None
try:
    openfold3_processor = OpenFold3JobProcessor(custom_ref_file)
except Exception as e:
    print(f"{chr(0x274c)} Failed to initialize processor: {e}")
    raise

print(f"\n{chr(0x1f504)} Processing CSV...")
try:
    jobs, summary_df = openfold3_processor.process_csv(csv_filename)
except Exception as e:
    print(f"{chr(0x274c)} CSV processing failed: {e}")
    raise

print("\n" + "=" * 60)
print(f"{chr(0x1f4cb)} JOB SUMMARY")
print("=" * 60)
print(summary_df.to_string(index=False))

if jobs:
    import json as _json_preview
    print("\n" + "=" * 60)
    print("JSON PREVIEW (first job)")
    print("=" * 60)
    preview_data = openfold3_processor._generate_query_json(jobs[0])
    full_preview = {"queries": {jobs[0]['name']: preview_data}}
    print(_json_preview.dumps(full_preview, indent=2))

error_jobs = summary_df[summary_df['status'] == 'ERROR']
if len(error_jobs) > 0:
    print(f"\n{chr(0x26a0)}  {len(error_jobs)} jobs have errors:")
    for _, row in error_jobs.iterrows():
        print(f"  - {row['jobname']}: {row['errors']}")

    proceed = input("\nProceed with valid jobs only? (yes/no): ").strip().lower()
    if proceed not in ['yes', 'y']:
        raise ValueError("Processing cancelled by user")

global_settings.update({
    'batch_jobs': jobs,
    'processor': openfold3_processor,
})

print("\n" + "=" * 60)
print(f"{chr(0x2705)} READY TO PROCESS {len(jobs)} JOBS")
print("=" * 60)
if global_settings.get('msa_folder'):
    print(f"Pre-computed MSAs: {global_settings['msa_folder']}")

print(f"\n{chr(0x1f4cc)} Next: Configure settings (Cell 5), then run (Cell 6)")

In [ ]:
#@title Cell 5: Configure OpenFold3 Prediction Parameters

print("=" * 60)
print("OPENFOLD3 PREDICTION PARAMETERS")
print("=" * 60)

# Core Model Parameters (CLI flags)
num_diffusion_samples = 5 #@param {type:"integer"}
#@markdown - **Diffusion samples**: Number of structure samples per seed. More = better chance of good structure. **Time**: Linear scaling. **VRAM**: Increases with more samples.

num_model_seeds = 1 #@param {type:"integer"}
#@markdown - **Number of model seeds**: How many random seeds to use (seeds generated starting from 42). Each seed produces an independent prediction. **Time**: Linear scaling. **VRAM**: No additional impact per seed.

#@markdown > **Note**: This MSA setting only applies when `msa_folder` is empty (Cell 4).
#@markdown > When pre-computed MSAs are provided, online MSA search is skipped entirely.

use_msa_server = True #@param {type:"boolean"}
#@markdown - **Use MSA server**: Generate MSAs using ColabFold server. Recommended for best accuracy. **Time**: Adds 30-120 seconds per protein sequence. **VRAM**: Minimal impact.

#@markdown ---
# Runner YAML settings (not available as CLI flags)
precision = "bf16-mixed" #@param ["bf16-mixed", "32-true"]
#@markdown - **Computation precision**: bf16-mixed is recommended for best speed/accuracy on A100. 32-true for maximum precision (slower, more VRAM). **Note**: Set via runner.yml (not a CLI flag).

structure_format = "cif" #@param ["cif", "pdb"]
#@markdown - **Output structure format**: mmCIF (cif) supports more metadata. PDB is more widely compatible. **Note**: Set via runner.yml (not a CLI flag).

low_mem = False #@param {type:"boolean"}
#@markdown - **Low memory preset**: Enable memory optimizations for tight VRAM situations. Reduces peak memory at cost of speed. **Note**: Set via runner.yml model preset.

skip_existing = False #@param {type:"boolean"}
#@markdown - **Skip existing**: Skip queries that already have output files. Useful for resuming interrupted batches. **Note**: Set via runner.yml.

#@markdown ---
#@markdown ### Interface Analysis (ipSAE_batch)
#@markdown Scores interface quality (ipSAE, ipTM, pDockQ) and generates visualizations for each prediction.

enable_ipsae = True #@param {type:"boolean"}
#@markdown - **Enable**: Run ipSAE interface analysis on each completed job.

ipsae_png = True #@param {type:"boolean"}
#@markdown - **PNG plots**: Matrix + ribbon visualizations per model (~10-30s/job).

ipsae_pdf = False #@param {type:"boolean"}
#@markdown - **PDF report**: Side-by-side model comparison per job (~20-60s/job).

ipsae_per_residue = False #@param {type:"boolean"}
#@markdown - **Per-residue CSV**: Detailed per-residue interface scores.

ipsae_per_contact = False #@param {type:"boolean"}
#@markdown - **Per-contact CSV**: Per-contact-pair scores (most detailed).

ipsae_pae_cutoff = 10.0 #@param {type:"number"}
#@markdown - **PAE cutoff**: PAE threshold for interface scoring (default: 10.0).

ipsae_dist_cutoff = 10.0 #@param {type:"number"}
#@markdown - **Distance cutoff**: CB-CB distance threshold in Angstroms (default: 10.0).

ipsae_select_residues = "" #@param {type:"string"}
#@markdown - **Select residues**: Focus on specific residues, e.g. `A:100-105,B:203`. Empty = all interfaces.

ipsae_batch_analysis = True #@param {type:"boolean"}
#@markdown - **Final batch comparison**: Combined CSV + interactive HTML across all jobs.

# Advanced Settings
print("\nAdvanced settings configured with defaults")

# Check if global_settings exists
if 'global_settings' not in globals():
    print("\nERROR: Please run Cell 4 (CSV Upload) first")
    raise ValueError("No batch jobs configured")

# Store all settings
global_settings['config'] = {
    'num_diffusion_samples': num_diffusion_samples,
    'num_model_seeds': num_model_seeds,
    'use_msa_server': use_msa_server,
    'precision': precision,
    'structure_format': structure_format,
    'low_mem': low_mem,
    'skip_existing': skip_existing,
    'enable_ipsae': enable_ipsae,
    'ipsae_png': ipsae_png,
    'ipsae_pdf': ipsae_pdf,
    'ipsae_per_residue': ipsae_per_residue,
    'ipsae_per_contact': ipsae_per_contact,
    'ipsae_pae_cutoff': ipsae_pae_cutoff,
    'ipsae_dist_cutoff': ipsae_dist_cutoff,
    'ipsae_select_residues': ipsae_select_residues,
    'ipsae_batch_analysis': ipsae_batch_analysis,
}

# Display summary
print("\n" + "=" * 60)
print("CONFIGURATION SUMMARY")
print("=" * 60)
print(f"CLI Settings:")
print(f"  - Diffusion samples: {num_diffusion_samples}")
print(f"  - Model seeds: {num_model_seeds} (seeds: {list(range(42, 42 + num_model_seeds))})")
print(f"  - Use MSA server: {use_msa_server}")

print(f"\nRunner YAML Settings:")
print(f"  - Precision: {precision}")
print(f"  - Structure format: {structure_format}")
print(f"  - Low memory: {low_mem}")
print(f"  - Skip existing: {skip_existing}")

total_predictions = num_model_seeds * num_diffusion_samples * len(global_settings['batch_jobs'])
print(f"\nTotal predictions: {num_model_seeds} seeds x {num_diffusion_samples} samples x {len(global_settings['batch_jobs'])} jobs = {total_predictions}")


print(f"\nInterface Analysis (ipSAE):")
print(f"  - Enabled: {enable_ipsae}")
if enable_ipsae:
    print(f"  - PNG: {ipsae_png} | PDF: {ipsae_pdf}")
    print(f"  - Per-residue: {ipsae_per_residue} | Per-contact: {ipsae_per_contact}")
    print(f"  - Cutoffs: PAE={ipsae_pae_cutoff}, dist={ipsae_dist_cutoff}")
    if ipsae_select_residues:
        print(f"  - Selected residues: {ipsae_select_residues}")
    print(f"  - Final batch comparison: {ipsae_batch_analysis}")

print(f"\nNext: Run Cell 6 to start batch predictions")

In [ ]:
#@title Cell 6: Run Batch Predictions with Calibration-Based Parallel GPU Scheduling
#@markdown **Runs job 1 alone to calibrate VRAM usage, then launches remaining jobs in parallel when GPU memory allows.**
#@markdown Results are automatically uploaded to Google Drive after each job completes.
%%time

import subprocess
import os
import zipfile
import time
import gc
import threading
import queue
import re
import json as _json
import shutil
from datetime import datetime


def get_unique_job_dir(job_name):
    """Return a unique job directory, appending _2, _3, etc. if exists."""
    if not os.path.exists(job_name):
        return job_name
    n = 2
    while os.path.exists(f"{job_name}_{n}"):
        n += 1
    return f"{job_name}_{n}"

def has_existing_output(job_name, search_dir="/content"):
    """Return True if job_name/ contains any .cif or .pdb structure files."""
    job_dir = os.path.join(search_dir, job_name)
    if not os.path.isdir(job_dir):
        return False
    for root, _, files in os.walk(job_dir):
        for f in files:
            if f.endswith(('.cif', '.pdb')):
                return True
    return False

# ============================================================
# VERIFY SETUP
# ============================================================
if 'global_settings' not in globals():
    print("ERROR: Please run the previous configuration cells first")
    raise ValueError("Configuration not found")

if not global_settings.get('batch_jobs'):
    print("ERROR: No batch jobs configured")
    print("   Please run Cell 4 to upload and process your CSV")
    raise ValueError("No jobs to process")

config = global_settings['config']
jobs = global_settings['batch_jobs']
processor = global_settings['processor']
drive = global_settings.get('drive')
gdrive_folder_name = global_settings.get('gdrive_folder_name', 'OpenFold3_Results')


# Check ipSAE_batch availability
ipsae_available = False
try:
    result = subprocess.run(["ipsae-batch", "--help"], capture_output=True, text=True, timeout=10)
    ipsae_available = (result.returncode == 0)
except Exception:
    pass
if config.get('enable_ipsae') and not ipsae_available:
    print("WARNING: ipSAE_batch not installed. Interface analysis will be skipped.")
# ============================================================
# PARALLEL EXECUTION SETTINGS
# ============================================================
enable_parallel = True #@param {type:"boolean"}
#@markdown - **Enable parallel execution**: Launch multiple jobs simultaneously when VRAM allows
#@markdown > **Note**: Jobs are automatically sorted largest-first for optimal calibration. The largest job calibrates VRAM, giving the most accurate per-token rate for parallel scheduling.

vram_limit = 0.85 #@param {type:"slider", min:0.6, max:0.95, step:0.05}
#@markdown - **VRAM limit**: Max fraction of GPU memory to use (0.85 = 85%). Jobs won't launch if this would be exceeded.

# ============================================================
# GPU VERIFICATION (CRITICAL: 32GB+ REQUIRED)
# ============================================================
print("=" * 60)
print("GPU VERIFICATION")
print("=" * 60)
gpu_available = False
total_gpu_vram = 0
try:
    import torch
    if torch.cuda.is_available():
        gpu_name = torch.cuda.get_device_name(0)
        gpu_memory_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
        total_gpu_vram = gpu_memory_gb
        print(f"GPU: {gpu_name}")
        print(f"   Total Memory: {gpu_memory_gb:.1f} GB")
        print(f"   VRAM limit ({vram_limit*100:.0f}%): {gpu_memory_gb * vram_limit:.1f} GB")

        # CRITICAL: Check for 32GB minimum
        if gpu_memory_gb < 30.0:
            print("\n" + "!" * 60)
            print("!!  CRITICAL ERROR: INSUFFICIENT GPU MEMORY  !!")
            print("!" * 60)
            print(f"   Detected: {gpu_memory_gb:.1f} GB VRAM")
            print(f"   Required: 32 GB minimum (A100 40GB or 80GB)")
            print(f"   OpenFold3 CANNOT run on {gpu_name}.")
            print(f"   Please change runtime: Runtime > Change runtime type > A100")
            print("!" * 60)
            raise RuntimeError(
                f"OpenFold3 requires 32GB+ VRAM. Current GPU ({gpu_name}) has only "
                f"{gpu_memory_gb:.1f} GB. Please switch to an A100 runtime."
            )

        test_tensor = torch.zeros(1).cuda()
        del test_tensor
        torch.cuda.synchronize()
        gpu_available = True

        try:
            result = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                                  capture_output=True, text=True, timeout=2)
            if result.returncode == 0:
                initial_mem = float(result.stdout.strip()) / 1024
                print(f"   Current Usage: {initial_mem:.2f} GB")
                print(f"   GPU monitoring ready")
            else:
                print(f"   WARNING: nvidia-smi not available, parallel mode disabled")
                enable_parallel = False
        except Exception as e:
            print(f"   WARNING: nvidia-smi test failed: {e}")
            enable_parallel = False
    else:
        print("WARNING: No GPU detected")
        print("   OpenFold3 requires an A100 GPU with 32GB+ VRAM")
        raise RuntimeError("No GPU detected. OpenFold3 requires an A100 GPU.")
except RuntimeError:
    raise
except Exception as e:
    print(f"WARNING: GPU test failed: {e}")
    gpu_available = False
    enable_parallel = False

# Precision check
if 'bf16' in config.get('precision', 'bf16-mixed'):
    print(f"\nUsing BF16 precision - optimal for A100")
else:
    print(f"\nUsing {config.get('precision', 'bf16-mixed')} precision")

# ============================================================
# GPU MONITOR CLASS
# ============================================================
class GPUMonitor:
    """Monitor GPU memory usage using nvidia-smi"""

    def __init__(self):
        self.monitoring = False
        self.peak_memory = 0
        self.current_memory = 0
        self.monitor_thread = None
        self._lock = threading.Lock()
        self.gpu_available = self._test_nvidia_smi()
        self._pid_registry = {}  # shell_pid -> {name, peak_gb}

    def _test_nvidia_smi(self):
        try:
            result = subprocess.run(['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                                  capture_output=True, text=True, timeout=2)
            return result.returncode == 0
        except:
            return False

    def _get_gpu_memory(self):
        if not self.gpu_available:
            return 0
        try:
            result = subprocess.run(
                ['nvidia-smi', '--query-gpu=memory.used', '--format=csv,noheader,nounits'],
                capture_output=True, text=True, timeout=2
            )
            if result.returncode == 0:
                return float(result.stdout.strip()) / 1024  # MB to GB
        except:
            pass
        return 0

    def _monitor_loop(self):
        while self.monitoring:
            mem = self._get_gpu_memory()
            # Per-PID VRAM tracking
            job_mem = {}
            with self._lock:
                need_pid = bool(self._pid_registry)
                shells = set(self._pid_registry.keys()) if need_pid else set()
            if need_pid:
                pid_mem = self._get_pid_mem()
                for gpu_pid, gb in pid_mem.items():
                    shell = self._find_job_for_gpu_pid(gpu_pid, shells)
                    if shell is not None:
                        job_mem[shell] = job_mem.get(shell, 0) + gb
            with self._lock:
                self.current_memory = mem
                if mem > self.peak_memory:
                    self.peak_memory = mem
                for spid, cur_gb in job_mem.items():
                    if spid in self._pid_registry:
                        if cur_gb > self._pid_registry[spid]['peak_gb']:
                            self._pid_registry[spid]['peak_gb'] = cur_gb
            time.sleep(0.5)

    def start(self):
        if not self.gpu_available:
            return
        with self._lock:
            self.monitoring = True
            self.peak_memory = 0
            self.current_memory = 0
        self.monitor_thread = threading.Thread(target=self._monitor_loop, daemon=True)
        self.monitor_thread.start()

    def stop(self):
        self.monitoring = False
        if self.monitor_thread:
            self.monitor_thread.join(timeout=2)
        return self.peak_memory

    def get_current(self):
        with self._lock:
            return self.current_memory

    def get_peak(self):
        with self._lock:
            return self.peak_memory

    def reset_peak(self):
        with self._lock:
            self.peak_memory = self.current_memory

    def _get_pid_mem(self):
        """Get per-PID GPU memory {pid: GB}."""
        try:
            r = subprocess.run(
                ['nvidia-smi', '--query-compute-apps=pid,used_gpu_memory', '--format=csv,noheader,nounits'],
                capture_output=True, text=True, timeout=2
            )
            if r.returncode == 0 and r.stdout.strip():
                result = {}
                for ln in r.stdout.strip().split('\n'):
                    parts = ln.strip().split(',')
                    if len(parts) >= 2:
                        try:
                            result[int(parts[0].strip())] = float(parts[1].strip()) / 1024
                        except ValueError:
                            pass
                return result
        except:
            pass
        return {}

    def _find_job_for_gpu_pid(self, gpu_pid, registered_shells):
        """Walk process tree up to find which registered job owns this GPU PID."""
        current = gpu_pid
        visited = set()
        while current > 1 and current not in visited:
            visited.add(current)
            if current in registered_shells:
                return current
            try:
                with open(f'/proc/{current}/status') as f:
                    for line in f:
                        if line.startswith('PPid:'):
                            current = int(line.split(':')[1].strip())
                            break
                    else:
                        break
            except (FileNotFoundError, PermissionError, ValueError):
                break
        return None

    def register_pid(self, shell_pid, job_name):
        """Register a job process PID for per-job VRAM tracking."""
        with self._lock:
            self._pid_registry[shell_pid] = {'name': job_name, 'peak_gb': 0.0}

    def unregister_pid(self, shell_pid):
        """Unregister and return peak VRAM (GB) for a job."""
        with self._lock:
            entry = self._pid_registry.pop(shell_pid, None)
            return entry['peak_gb'] if entry else 0.0

    def get_pid_peak(self, shell_pid):
        """Get peak VRAM (GB) for a specific registered job."""
        with self._lock:
            entry = self._pid_registry.get(shell_pid)
            return entry['peak_gb'] if entry else 0.0


gpu_monitor = GPUMonitor()

def clear_gpu_cache():
    if not gpu_available:
        return
    try:
        import torch
        torch.cuda.empty_cache()
        gc.collect()
    except:
        pass

# ============================================================
# GOOGLE DRIVE HELPERS
# ============================================================
def get_or_create_folder(drive, folder_name, parent_id='root'):
    if not drive:
        return None
    try:
        file_list = drive.ListFile({
            'q': f"'{parent_id}' in parents and title='{folder_name}' and mimeType='application/vnd.google-apps.folder' and trashed=false"
        }).GetList()
        if file_list:
            print(f"Found existing folder: {folder_name}")
            return file_list[0]['id']
        else:
            folder = drive.CreateFile({
                'title': folder_name,
                'mimeType': 'application/vnd.google-apps.folder',
                'parents': [{'id': parent_id}]
            })
            folder.Upload()
            print(f"Created new folder: {folder_name}")
            return folder['id']
    except Exception as e:
        print(f"ERROR with folder '{folder_name}': {e}")
        return None

def upload_to_gdrive(drive, file_path, folder_id, job_name):
    if not drive or not os.path.exists(file_path):
        return None
    try:
        uploaded_file = drive.CreateFile({
            'title': os.path.basename(file_path),
            'parents': [{'id': folder_id}]
        })
        uploaded_file.SetContentFile(file_path)
        uploaded_file.Upload()
        file_url = f"https://drive.google.com/file/d/{uploaded_file['id']}/view"
        print(f"  Uploaded to Google Drive: {file_url}")
        # Close file handle to prevent ResourceWarning
        if hasattr(uploaded_file, 'content') and uploaded_file.content:
            try:
                uploaded_file.content.close()
            except Exception:
                pass
        return file_url
    except Exception as e:
        print(f"  WARNING: Upload failed: {e}")
        return None

# ============================================================
# UTILITY FUNCTIONS
# ============================================================
def read_error_files(job_dir, job_name):
    error_info = []
    error_dirs = [
        os.path.join(job_dir, "errors"),
        os.path.join(job_dir, job_name, "errors"),
        os.path.join(job_dir, "logs"),
        os.path.join(job_dir, job_name, "logs"),
    ]
    for error_dir in error_dirs:
        if os.path.exists(error_dir):
            try:
                for f in os.listdir(error_dir):
                    if f.endswith(('.txt', '.log', '.json')):
                        try:
                            with open(os.path.join(error_dir, f), 'r') as fh:
                                content = fh.read()
                                # Show last 800 chars for log files (errors at the end)
                                if f.endswith('.log') and len(content) > 800:
                                    content = '...\n' + content[-800:]
                                else:
                                    content = content[:800]
                                error_info.append({'file': f, 'content': content})
                        except:
                            pass
            except:
                pass
    return error_info

def find_predictions_directory(job_dir, job_name):
    possible_paths = [
        job_dir,
        os.path.join(job_dir, job_name),
        os.path.join(job_dir, "predictions"),
        os.path.join(job_dir, job_name, "predictions"),
        os.path.join(job_dir, "predictions", job_name),
        os.path.join(job_dir, "output"),
        os.path.join(job_dir, job_name, "output"),
    ]
    for path in possible_paths:
        if os.path.exists(path):
            try:
                for root, dirs, files_list in os.walk(path):
                    for file in files_list:
                        if file.endswith(('.cif', '.mmcif')):
                            return root
            except:
                continue
    return None

def parse_openfold3_progress(line):
    keywords = ['progress', 'step', 'recycling', 'diffusion', 'sample',
                'completed', 'saving', 'msa', 'model', 'generating',
                'loading', 'processing', 'predicting', 'running',
                'confidence', 'output', 'writing']
    line_lower = line.lower()
    return any(kw in line_lower for kw in keywords)

def count_job_tokens(job, processor):
    total_tokens = 0
    for seq in job.get('sequences', []):
        seq_type = seq.get('type', '').lower()
        if seq_type in ['protein', 'dna', 'rna']:
            total_tokens += len(seq.get('sequence', ''))
            modifications = seq.get('modifications')
            if modifications:
                for mod in modifications:
                    mod_code = mod.get('ccd', '')
                    if seq_type == 'protein' and mod_code in processor.ptm_lookup:
                        total_tokens += 10
                    elif seq_type == 'dna' and mod_code in processor.dna_mod_lookup:
                        total_tokens += 10
                    elif seq_type == 'rna' and mod_code in processor.rna_mod_lookup:
                        total_tokens += 10
        elif seq_type == 'ligand':
            ccd_code = seq.get('ccd')
            smiles = seq.get('smiles')
            if ccd_code:
                if ccd_code in processor.ligand_lookup:
                    total_tokens += 20
                elif ccd_code in processor.ion_lookup:
                    total_tokens += 1
                elif ccd_code in processor.glycan_lookup:
                    total_tokens += 15
                else:
                    total_tokens += 20
            elif smiles:
                total_tokens += 30
    return max(total_tokens, 1)

# ============================================================
# NON-BLOCKING PROCESS OUTPUT READER
# ============================================================

# ============================================================
# MSA FOLDER LOOKUP HELPER
# ============================================================
def find_precomputed_msas(job, msa_folder):
    """Look up pre-computed MSA files for ALL protein chains in a job.

    All-or-nothing: returns MSA paths only if EVERY protein chain has a
    pre-computed MSA file. If any chain is missing, returns empty dict
    and the job falls back to online MSA search entirely.
    """
    if not msa_folder or not os.path.isdir(msa_folder):
        return {}

    job_name = job['name']
    found = {}
    missing = []

    seen = set()
    protein_chains = []
    for seq in job.get('sequences', []):
        if seq.get('type', '').lower() != 'protein':
            continue
        chain_name = seq.get('name', '')
        if not chain_name or chain_name in seen:
            continue
        seen.add(chain_name)
        protein_chains.append(chain_name)

    if not protein_chains:
        return {}

    available_files = {}
    for fname in os.listdir(msa_folder):
        if fname.endswith('_paired.a3m'):
            normalized = fname.lower().replace('-', '_')
            available_files[normalized] = fname

    for chain_name in protein_chains:
        expected = f"{job_name}_{chain_name}_paired.a3m"
        exact_path = os.path.join(msa_folder, expected)
        if os.path.isfile(exact_path):
            found[chain_name] = exact_path
        else:
            norm_key = expected.lower().replace('-', '_')
            if norm_key in available_files:
                found[chain_name] = os.path.join(msa_folder, available_files[norm_key])
            else:
                missing.append(chain_name)

    if missing:
        print(f"   Pre-computed MSAs missing for: {missing} -- using online MSA for all chains")
        return {}

    print(f"   Pre-computed MSAs found for all {len(found)} chain(s): {list(found.keys())}")
    return found

def sanitize_a3m_for_openfold3(src_path, dst_path):
    """Copy a3m file, filtering rows with mismatched alignment length.

    OpenFold3's _msa_list_to_np requires all sequences to have the same
    length after stripping lowercase insertion characters. ColabFold
    occasionally returns rows that violate this for short sequences.
    """
    import string
    deletion_table = str.maketrans("", "", string.ascii_lowercase)

    with open(src_path) as f:
        content = f.read()

    entries = []
    current_header = None
    current_seq_parts = []

    for line in content.strip().split('\n'):
        line = line.rstrip()
        if not line or line.startswith('#'):
            continue
        if line.startswith('>'):
            if current_header is not None:
                entries.append((current_header, ''.join(current_seq_parts)))
            current_header = line
            current_seq_parts = []
        else:
            current_seq_parts.append(line)
    if current_header is not None:
        entries.append((current_header, ''.join(current_seq_parts)))

    if not entries:
        import shutil
        shutil.copy2(src_path, dst_path)
        return 0

    query_len = len(entries[0][1].translate(deletion_table))
    kept = []
    dropped = 0
    for header, seq in entries:
        if len(seq.translate(deletion_table)) == query_len:
            kept.append((header, seq))
        else:
            dropped += 1

    with open(dst_path, 'w') as f:
        for header, seq in kept:
            f.write(f"{header}\n{seq}\n")

    return dropped

# Known harmless warnings to suppress from subprocess output
_SUPPRESS_PATTERNS = [
    'Non-SM100f kernel expects bias to be float32',
    'builtin type swigvarlink has no __module__',
    'builtin type SwigPyPacked has no __module__',
    'builtin type SwigPyObject has no __module__',
    'datetime.datetime.utcnow() is deprecated',
]

def _is_suppressed_warning(line):
    """Check if a line matches known harmless warning patterns."""
    return any(pat in line for pat in _SUPPRESS_PATTERNS)

def _reader_thread(pipe, output_queue):
    """Read lines from a pipe into a thread-safe queue."""
    try:
        for line in iter(pipe.readline, ''):
            output_queue.put(line.rstrip('\n'))
    except:
        pass
    finally:
        try:
            pipe.close()
        except:
            pass

# ============================================================
# BUILD OPENFOLD3 COMMAND
# ============================================================
def _write_runner_yaml(job_dir, current_config):
    """Write a runner.yml for settings not available as CLI flags."""
    import yaml
    runner_config = {}

    # Precision
    precision = current_config.get('precision', 'bf16-mixed')
    runner_config['pl_trainer_args'] = {'precision': precision}

    # Output format
    structure_format = current_config.get('structure_format', 'cif')
    runner_config['output_writer_settings'] = {'structure_format': structure_format}

    # Low memory preset
    presets = ['predict', 'pae_enabled']
    if current_config.get('low_mem', False):
        presets.append('low_mem')

    # Kernel configuration (CRITICAL — see GitHub issues #17, #23, #62)
    # DeepSpeed evoformer attention ALWAYS disabled on Colab:
    #   it requires CUTLASS_PATH pointing to a source tree for JIT compilation,
    #   which is impractical in Colab. Without it, prediction fails with
    #   "Unable to JIT load the evoformer_attn extension".
    # cuEquivariance: enabled only if preflight test passed in Cell 2.
    # When both are off: falls back to standard PyTorch attention (slower but works).
    # NOTE: If a future Colab/OpenFold3 update fixes cuEquivariance or
    # DeepSpeed compatibility, the preflight test in Cell 2 will detect it.
    cueq_ok = global_settings.get('cueq_available', False)
    runner_config['model_update'] = {
        'presets': presets,
        'custom': {
            'settings': {
                'memory': {
                    'eval': {
                        'use_deepspeed_evo_attention': False,
                        'use_cueq_triangle_kernels': cueq_ok,
                    }
                }
            }
        }
    }

    # Disable internal MSA cache cleanup — parallel jobs share /tmp/of3_colabfold_msas
    # and internal cleanup after one job wipes files for sibling jobs (issue #39)
    runner_config['msa_computation_settings'] = {'cleanup_msa_dir': False}

    # Skip existing
    if current_config.get('skip_existing', False):
        runner_config['experiment_settings'] = {'skip_existing': True}

    yaml_path = os.path.join(job_dir, 'runner.yml')
    with open(yaml_path, 'w') as f:
        yaml.dump(runner_config, f, default_flow_style=False)
    return yaml_path


def build_openfold3_cmd(json_file, job_dir, current_config):
    """Build the run_openfold predict command."""
    # Write runner.yml for non-CLI settings (precision, output format, presets)
    runner_yaml = _write_runner_yaml(job_dir, current_config)

    # Explicit checkpoint path (safety belt — avoids runtime input() prompt)
    ckpt_path = os.path.expanduser("~/.openfold3/of3-p2-155k.pt")

    cmd_parts = [
        "run_openfold", "predict",
        f"--query_json={json_file}",
        f"--output_dir={job_dir}",
        f"--runner_yaml={runner_yaml}",
        f"--inference_ckpt_path={ckpt_path}",
    ]

    # Diffusion samples (CLI flag)
    num_samples = current_config.get('num_diffusion_samples', 5)
    cmd_parts.append(f"--num_diffusion_samples={num_samples}")

    # Model seeds (CLI flag)
    num_seeds = current_config.get('num_model_seeds', 1)
    if num_seeds > 1:
        cmd_parts.append(f"--num_model_seeds={num_seeds}")

    # MSA server (CLI flag, default True)
    # Only disable when MSAs were actually injected for this job, or user explicitly disabled
    if not current_config.get('use_msa_server', True) or current_config.get('_msas_injected'):
        cmd_parts.append("--use_msa_server=False")

    # Templates always disabled (known bug: issue #101 IndexError)
    cmd_parts.append("--use_templates=False")

    return " ".join(cmd_parts)


# ============================================================
# ipSAE INTERFACE ANALYSIS
# ============================================================
def run_ipsae_analysis(job_dir, job_name):
    """Run ipSAE per-job analysis. Returns True on success, False on failure."""
    if not ipsae_available or not config.get('enable_ipsae'):
        return False

    ipsae_output = os.path.join(job_dir, "ipsae_analysis")
    os.makedirs(ipsae_output, exist_ok=True)

    cmd_parts = [
        "ipsae-batch", job_dir,
        "--backend", "openfold3",
        "--output_dir", ipsae_output,
        "--pae_cutoff", str(config.get('ipsae_pae_cutoff', 10.0)),
        "--dist_cutoff", str(config.get('ipsae_dist_cutoff', 10.0)),
        "--workers", "1",
        "--cores", "1",
    ]
    if config.get('ipsae_png'):
        cmd_parts.append("--png")
    if config.get('ipsae_pdf'):
        cmd_parts.append("--pdf")
    if config.get('ipsae_per_residue'):
        cmd_parts.append("--per_residue")
    if config.get('ipsae_per_contact'):
        cmd_parts.append("--per_contact")
    if config.get('ipsae_select_residues'):
        cmd_parts.extend(["--select-residues", config['ipsae_select_residues']])

    print(f"   Running ipSAE analysis...")
    try:
        result = subprocess.run(cmd_parts, capture_output=True, text=True, timeout=300)
        if result.returncode == 0:
            n_files = sum(1 for f in os.listdir(ipsae_output) if not f.startswith('.'))
            print(f"   ipSAE complete: {n_files} output files")
            return True
        else:
            print(f"   ipSAE failed (rc={result.returncode}): {result.stderr[:200]}")
            return False
    except subprocess.TimeoutExpired:
        print(f"   ipSAE timed out (300s)")
        return False
    except Exception as e:
        print(f"   ipSAE error: {e}")
        return False

# ============================================================
# PROCESS COMPLETED JOB (zip + upload + collect results)
# ============================================================
def process_completed_job(job_name, job_dir, json_file, job_time, peak_gpu, drive, gdrive_folder_id):
    """Find outputs, zip, upload to GDrive. Returns result dict."""
    predictions_dir = find_predictions_directory(job_dir, job_name)

    if not predictions_dir:
        print(f"\nERROR: No structure files (.cif/.mmcif) found for {job_name}")
        error_files = read_error_files(job_dir, job_name)
        if error_files:
            for err_info in error_files[:2]:
                print(f"   Error log {err_info['file']}:")
                print(f"   {err_info['content']}")
        # Show directory structure
        print(f"   Directory structure of {job_dir}:")
        for root, dirs, files_list in os.walk(job_dir):
            level = root.replace(job_dir, '').count(os.sep)
            indent = '   ' * (level + 1)
            print(f"{indent}{os.path.basename(root)}/")
            for file in files_list[:10]:
                print(f"{indent}  {file}")
        return {
            'success': False, 'name': job_name,
            'error': 'No structure files generated',
            'time': job_time, 'gpu_peak': peak_gpu
        }

    structure_files = [f for f in os.listdir(predictions_dir) if f.endswith(('.cif', '.mmcif'))]

    if not structure_files:
        print(f"\nERROR: No structure files in predictions directory for {job_name}")
        return {
            'success': False, 'name': job_name,
            'error': 'No structure files in predictions directory',
            'time': job_time, 'gpu_peak': peak_gpu
        }

    print(f"   {len(structure_files)} structure files generated")
    for f in sorted(structure_files)[:3]:
        print(f"      {f}")
    if len(structure_files) > 3:
        print(f"      ... and {len(structure_files) - 3} more")

    # Look for confidence JSON files
    confidence_files = []
    for root, dirs, files_list in os.walk(job_dir):
        for f in files_list:
            if 'confidence' in f.lower() and f.endswith('.json'):
                confidence_files.append(os.path.join(root, f))

    if confidence_files:
        print(f"   {len(confidence_files)} confidence file(s) found")
        for cf in confidence_files[:2]:
            try:
                with open(cf, 'r') as fh:
                    conf_data = _json.load(fh)
                if isinstance(conf_data, dict):
                    for key in ['plddt', 'ptm', 'iptm', 'ranking_score']:
                        if key in conf_data:
                            val = conf_data[key]
                            if isinstance(val, (int, float)):
                                print(f"      {key}: {val:.4f}")
                            elif isinstance(val, list) and len(val) > 0:
                                avg_val = sum(val) / len(val)
                                print(f"      {key} (mean): {avg_val:.4f}")
            except:
                pass

    # --- ipSAE interface analysis (before zipping) ---
    ipsae_ran = False
    try:
        ipsae_ran = run_ipsae_analysis(job_dir, job_name)
    except Exception as e:
        print(f"   ipSAE error (non-fatal): {e}")

    # Create ZIP
    zip_filename = f"{job_name}_results.zip"
    zip_path = os.path.join(job_dir, zip_filename)
    try:
        with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
            added_arcnames = set()
            for root, dirs, files_list in os.walk(predictions_dir):
                for file in files_list:
                    file_path = os.path.join(root, file)
                    arcname = os.path.relpath(file_path, job_dir)
                    zipf.write(file_path, arcname)
                    added_arcnames.add(arcname)
            # Include confidence files (skip if already added from predictions_dir walk)
            for cf in confidence_files:
                arcname = os.path.relpath(cf, job_dir)
                if arcname not in added_arcnames:
                    zipf.write(cf, arcname)
                    added_arcnames.add(arcname)
            # Include query JSON
            json_arcname = os.path.basename(json_file)
            if json_arcname not in added_arcnames:
                zipf.write(json_file, json_arcname)
                # Add environment log for traceability
                if os.path.isfile("/content/environment.txt"):
                    zipf.write("/content/environment.txt", "environment.txt")
            # Add ipSAE analysis files
            ipsae_dir = os.path.join(job_dir, "ipsae_analysis")
            if os.path.isdir(ipsae_dir):
                for root, dirs, files_list in os.walk(ipsae_dir):
                    for f in files_list:
                        fp = os.path.join(root, f)
                        arcname = os.path.join("ipsae_analysis", os.path.relpath(fp, ipsae_dir))
                        zipf.write(fp, arcname)
        print(f"   Created: {zip_filename}")
    except Exception as e:
        print(f"   WARNING: Failed to create ZIP: {e}")

    # Upload to GDrive
    file_url = None
    if drive and gdrive_folder_id:
        file_url = upload_to_gdrive(drive, zip_path, gdrive_folder_id, job_name)
    else:
        print(f"   Saved locally: {zip_path}")

    return {
        'success': True, 'name': job_name,
        'structures': len(structure_files),
        'time': job_time, 'url': file_url,
        'gpu_peak': peak_gpu,
        'ipsae_ran': ipsae_ran,
    }

# ============================================================
# SEQUENTIAL RUNNER
# ============================================================
def run_single_prediction_sequential(job, job_idx, total_jobs, retry_with_fp32=False):
    """Run a single OpenFold3 prediction sequentially with GPU monitoring."""
    job_start = time.time()

    print("\n" + "=" * 60)
    print(f"Job {job_idx}/{total_jobs}: {job['name']}")
    if retry_with_fp32:
        print("   RETRY with fp32 precision")
    print("=" * 60)

    # Skip existing check
    if config.get('skip_existing', False) and has_existing_output(job['name']):
        print(f"  \u23ed Skipping {job['name']}: output already exists")
        return {'success': True, 'name': job['name'], 'skipped': True,
                'structures': 0, 'time': 0, 'gpu_peak': 0,
                'tokens': count_job_tokens(job, processor)}

    gpu_monitor.start()
    time.sleep(0.5)
    initial_mem = gpu_monitor.get_current()
    print(f"GPU Memory (start): {initial_mem:.2f} GB")

    job_name = job['name']
    job_dir = get_unique_job_dir(job_name)
    os.makedirs(job_dir, exist_ok=True)

    # Generate OpenFold3 JSON query
    query_data = processor._generate_query_json(job)

    # Inject pre-computed MSA paths if available
    # v0.4.0 requires files named colabfold_main.a3m / colabfold_paired.a3m
    # inside per-chain directories (filename whitelist in parse_msas_direct)
    _msas_injected = False
    msa_folder_val = global_settings.get('msa_folder', '')
    if msa_folder_val:
        precomp_msas = find_precomputed_msas(job, msa_folder_val)
        if precomp_msas:
            import shutil as _sh
            id_to_name = {}
            for seq in job.get('sequences', []):
                if seq.get('type', '').lower() == 'protein' and seq.get('name'):
                    id_to_name[seq.get('id', '')] = seq['name']
            for chain_entry in query_data.get('chains', []):
                if chain_entry.get('molecule_type') != 'protein':
                    continue
                for cid in chain_entry.get('chain_ids', []):
                    chain_name = id_to_name.get(cid)
                    if chain_name and chain_name in precomp_msas:
                        paired_src = precomp_msas[chain_name]
                        ind_dir = os.path.join(os.path.dirname(msa_folder_val), 'individual')
                        ind_file = os.path.join(ind_dir, f"{chain_name}.a3m")
                        main_src = ind_file if os.path.isfile(ind_file) else paired_src
                        # Create ColabFold-style directory structure
                        chain_msa_dir = os.path.join(job_dir, "precomputed_msas", chain_name)
                        os.makedirs(chain_msa_dir, exist_ok=True)
                        dropped = sanitize_a3m_for_openfold3(main_src, os.path.join(chain_msa_dir, "colabfold_main.a3m"))
                        _sh.copy2(paired_src, os.path.join(chain_msa_dir, "colabfold_paired.a3m"))
                        chain_entry['main_msa_file_paths'] = [chain_msa_dir]
                        chain_entry['paired_msa_file_paths'] = [chain_msa_dir]
                        with open(os.path.join(chain_msa_dir, "colabfold_main.a3m")) as _mf:
                            n_seqs = sum(1 for _line in _mf if _line.startswith('>'))
                        print(f"      {chain_name}: {n_seqs} MSA sequences" + (f" ({dropped} malformed rows removed)" if dropped else ""))
                        break
            _msas_injected = True
    full_query = {"queries": {job_name: query_data}}

    json_file = os.path.join(job_dir, f"{job_name}_query.json")
    with open(json_file, 'w') as f:
        _json.dump(full_query, f, indent=2)

    print(f"Query JSON: {json_file}")
    print(f"Chains: {len(query_data['chains'])}")
    for chain in query_data['chains']:
        cids = ','.join(chain['chain_ids'])
        mtype = chain['molecule_type']
        if mtype in ('protein', 'dna', 'rna'):
            seqlen = len(chain.get('sequence', ''))
            print(f"   [{cids}] {mtype} ({seqlen} residues)")
        else:
            smi = chain.get('smiles', chain.get('ccd_code', '?'))
            print(f"   [{cids}] {mtype} ({smi[:40]})")

    current_config = config.copy()
    if _msas_injected:
        current_config['_msas_injected'] = True
    if retry_with_fp32:
        current_config['precision'] = '32-true'
        print(f"   Changed precision to fp32 (32-true)")

    cmd = build_openfold3_cmd(json_file, job_dir, current_config)
    print(f"Command: {cmd}")

    # Clean MSA cache between sequential jobs to prevent stale file conflicts (issue #39)
    # Safe in sequential mode since only one job runs at a time
    shutil.rmtree('/tmp/of3_colabfold_msas', ignore_errors=True)

    print(f"\nRunning OpenFold3 prediction...")

    prediction_failed = False
    error_messages = []

    # Phase tracking
    phase_times = {}
    current_phase = "initialization"
    phase_start = time.time()

    try:
        process = subprocess.Popen(
            cmd, shell=True,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, universal_newlines=True
        )

        # Non-blocking reader
        output_q = queue.Queue()
        reader_t = threading.Thread(target=_reader_thread, args=(process.stdout, output_q), daemon=True)
        reader_t.start()

        print("\nProgress:")
        while process.poll() is None:
            while not output_q.empty():
                try:
                    line = output_q.get_nowait()
                except queue.Empty:
                    break
                if not line:
                    continue

                # Track errors
                if 'error' in line.lower() and ('nan' in line.lower() or 'inf' in line.lower() or 'oom' in line.lower()):
                    error_messages.append(line)
                    prediction_failed = True
                elif 'out of memory' in line.lower():
                    error_messages.append(line)
                    prediction_failed = True
                elif 'keyerror' in line.lower() or ('error' in line.lower() and 'msa' in line.lower()):
                    error_messages.append(line)
                    prediction_failed = True

                # Track phases
                ll = line.lower()
                if 'msa' in ll and current_phase != "msa":
                    phase_times[current_phase] = time.time() - phase_start
                    current_phase = "msa"; phase_start = time.time()
                elif 'recycling' in ll and current_phase != "recycling":
                    phase_times[current_phase] = time.time() - phase_start
                    current_phase = "recycling"; phase_start = time.time()
                elif 'diffusion' in ll and current_phase != "diffusion":
                    phase_times[current_phase] = time.time() - phase_start
                    current_phase = "diffusion"; phase_start = time.time()
                elif ('saving' in ll or 'writing' in ll) and current_phase != "saving":
                    phase_times[current_phase] = time.time() - phase_start
                    current_phase = "saving"; phase_start = time.time()

                # Print progress
                if parse_openfold3_progress(line):
                    cur_gpu = gpu_monitor.get_current()
                    if cur_gpu > 1.0:
                        print(f"   {line[:300]} [GPU: {cur_gpu:.1f} GB]")
                    else:
                        print(f"   {line[:300]}")
                elif ('error' in line.lower() or 'warning' in line.lower()) and not _is_suppressed_warning(line):
                    print(f"   WARNING: {line[:300]}")

            time.sleep(0.5)

        # Drain remaining output
        while not output_q.empty():
            try:
                line = output_q.get_nowait()
                if line and (parse_openfold3_progress(line) or 'error' in line.lower()):
                    print(f"   {line[:300]}")
            except queue.Empty:
                break

        phase_times[current_phase] = time.time() - phase_start
        return_code = process.wait(timeout=60)
        peak_gpu_memory = gpu_monitor.stop()

        print(f"\nGPU Memory: Peak={peak_gpu_memory:.2f} GB, Final={gpu_monitor.get_current():.2f} GB")

        if return_code != 0:
            print(f"\nERROR: Prediction failed (return code: {return_code})")
            error_files = read_error_files(job_dir, job_name)
            if error_files:
                for err_info in error_files[:2]:
                    print(f"   {err_info['file']}: {err_info['content'][:200]}")
            return {
                'success': False, 'name': job_name,
                'error': f"Process exited with code {return_code}",
                'error_messages': error_messages,
                'time': time.time() - job_start, 'gpu_peak': peak_gpu_memory,
                'can_retry': prediction_failed and not retry_with_fp32
            }

    except subprocess.TimeoutExpired:
        process.kill()
        peak_gpu_memory = gpu_monitor.stop()
        return {
            'success': False, 'name': job_name, 'error': "Timeout",
            'time': time.time() - job_start, 'gpu_peak': peak_gpu_memory, 'can_retry': False
        }
    except Exception as e:
        peak_gpu_memory = gpu_monitor.stop()
        return {
            'success': False, 'name': job_name, 'error': str(e),
            'time': time.time() - job_start, 'gpu_peak': peak_gpu_memory, 'can_retry': False
        }

    # Print timing
    if phase_times:
        total_phase_time = sum(phase_times.values())
        print(f"\nTiming:")
        for phase, duration in phase_times.items():
            if duration > 0.1:
                pct = (duration / total_phase_time) * 100 if total_phase_time > 0 else 0
                print(f"   - {phase.capitalize()}: {duration:.1f}s ({pct:.0f}%)")

    job_time = time.time() - job_start
    result = process_completed_job(job_name, job_dir, json_file, job_time, peak_gpu_memory, drive, gdrive_folder_id)
    result['can_retry'] = prediction_failed and not retry_with_fp32
    if 'tokens' not in result:
        result['tokens'] = count_job_tokens(job, processor)
    print(f"\nTotal: {job_time:.1f}s ({job_time/60:.1f} min)")
    return result

# ============================================================
# SETUP GOOGLE DRIVE
# ============================================================
gdrive_folder_id = None
if drive:
    print("\n" + "=" * 60)
    print("SETTING UP GOOGLE DRIVE")
    print("=" * 60)
    gdrive_folder_id = get_or_create_folder(drive, gdrive_folder_name)
    if gdrive_folder_id:
        print(f"Results will be uploaded to: {gdrive_folder_name}")
    else:
        print("WARNING: Could not setup Drive folder - results will be saved locally only")
        drive = None
else:
    print("\n" + "=" * 60)
    print("NO GOOGLE DRIVE - LOCAL ONLY")
    print("=" * 60)
    print("WARNING: Google Drive not configured in Cell 3")

# ============================================================
# PRINT BATCH INFO
# ============================================================
print("\n" + "=" * 60)
print(f"STARTING BATCH PROCESSING: {len(jobs)} JOBS")
print("=" * 60)
print(f"Configuration:")
print(f"  - Model seeds: {config.get('num_model_seeds', 1)} (starting from 42)")
print(f"  - Use MSA server: {config.get('use_msa_server', True)}")

cueq_ok = global_settings.get('cueq_available', False)
print(f"  - Attention kernel: {'cuEquivariance' if cueq_ok else 'standard PyTorch (DeepSpeed disabled)'}")
print(f"  - Precision: {config.get('precision', 'bf16-mixed')}")
print(f"  - Output format: {config.get('structure_format', 'cif')}")
print(f"  - Parallel mode: {enable_parallel}")
print(f"\nStarted: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# Sort jobs largest-first for optimal calibration accuracy
jobs = sorted(jobs, key=lambda j: count_job_tokens(j, processor), reverse=True)
print(f"Jobs sorted by size (largest first):")
for _j in jobs:
    print(f"  {_j['name']}: {count_job_tokens(_j, processor)} tokens")

# ============================================================
# SINGLE JOB SHORTCUT
# ============================================================
if len(jobs) == 1:
    enable_parallel = False

# Force sequential when using online MSA server (ColabFold can't handle concurrent requests)
if enable_parallel and not global_settings.get('msa_folder') and config.get('use_msa_server', True):
    enable_parallel = False
    print("\n   Parallel disabled: ColabFold server cannot handle concurrent requests")
    print("   Tip: Use the MSA Colab to pre-compute MSAs for parallel execution")

# ============================================================
# MAIN EXECUTION
# ============================================================
# Clean MSA cache once before batch (prevents stale files from previous runs)
# Individual job cleanup is disabled (cleanup_msa_dir: false in runner.yml)
# to prevent parallel jobs from wiping files used by sibling jobs (issue #39)
shutil.rmtree('/tmp/of3_colabfold_msas', ignore_errors=True)

batch_start_time = time.time()
completed_jobs = []
failed_jobs = []
retried_jobs = []

if not enable_parallel or not gpu_available:
    # --------------------------------------------------------
    # SEQUENTIAL MODE
    # --------------------------------------------------------
    if not enable_parallel:
        print(f"\nRunning in SEQUENTIAL mode (parallel disabled)")
    else:
        print(f"\nRunning in SEQUENTIAL mode (no GPU monitoring)")

    for job_idx, job in enumerate(jobs, 1):
        result = run_single_prediction_sequential(job, job_idx, len(jobs))

        if result['success']:
            completed_jobs.append(result)
        else:
            if result.get('can_retry'):
                print(f"\nRetrying {job['name']} with fp32 precision...")
                retried_jobs.append(job['name'])
                clear_gpu_cache()
                time.sleep(2)
                result = run_single_prediction_sequential(job, job_idx, len(jobs), retry_with_fp32=True)
                if result['success']:
                    completed_jobs.append(result)
                else:
                    failed_jobs.append(result)
            else:
                failed_jobs.append(result)

        clear_gpu_cache()
        time.sleep(1)

else:
    # --------------------------------------------------------
    # CALIBRATION-BASED PARALLEL MODE
    # --------------------------------------------------------
    print(f"\n" + "=" * 60)
    print(f"CALIBRATION-BASED PARALLEL MODE")
    print("=" * 60)
    print(f"Phase 1: Run job 1 alone to measure actual VRAM usage")
    print(f"Phase 2: Use measurement to estimate remaining jobs")
    print(f"Phase 3: Launch jobs in parallel when VRAM allows")
    print("=" * 60)

    # --- PHASE 1: CALIBRATION (try jobs one by one until one succeeds) ---
    print(f"\n{chr(0x1f4cf)} PHASE 1: CALIBRATION")
    print(f"Running jobs one at a time until one succeeds to measure VRAM...")

    cal_result = None
    cal_peak_vram = 0
    cal_tokens = 0
    calibration_idx = -1

    skipped_in_cal = 0
    for cal_idx, cal_job in enumerate(jobs):
        cal_job_name = cal_job['name']

        # Skip already-completed jobs -- they give no VRAM data
        if config.get('skip_existing', False) and has_existing_output(cal_job_name):
            print(f"  \u23ed Skipping {cal_job_name} (already done)")
            completed_jobs.append({'name': cal_job_name, 'skipped': True, 'success': True,
                                   'structures': 0, 'time': 0, 'gpu_peak': 0,
                                   'tokens': count_job_tokens(cal_job, processor)})
            skipped_in_cal += 1
            continue

        cal_tokens = count_job_tokens(cal_job, processor)
        print(f"\n   Calibration attempt {cal_idx + 1}/{len(jobs)}: {cal_job['name']} ({cal_tokens} tokens)")

        cal_result = run_single_prediction_sequential(cal_job, cal_idx + 1, len(jobs))

        if cal_result['success']:
            completed_jobs.append(cal_result)
            calibration_idx = cal_idx
            cal_peak_vram = cal_result.get('gpu_peak', 0)
            print(f"\n   {chr(0x2705)} Calibration succeeded on job '{cal_job['name']}' -- peak VRAM: {cal_peak_vram:.2f} GB")
            break
        else:
            failed_jobs.append(cal_result)
            print(f"   {chr(0x26a0)}  Job '{cal_job['name']}' failed -- trying next job for calibration...")
            clear_gpu_cache()
            time.sleep(2)
            continue

    remaining_jobs = jobs[calibration_idx + 1:] if calibration_idx >= 0 else []

    if calibration_idx < 0:
        if skipped_in_cal == len(jobs):
            print(f"\n{chr(0x2705)} All {len(jobs)} jobs already completed. Nothing to run.")
        else:
            print(f"\n{chr(0x274c)} ALL {len(jobs)} jobs failed. No successful calibration possible.")
    elif not remaining_jobs:
        print("\nOnly one job - calibration complete, nothing more to run.")
    elif cal_peak_vram <= 0:
        # No VRAM data - fall back to sequential
        print(f"\nWARNING: Could not measure VRAM during calibration. Falling back to sequential.")
        for job_idx, job in enumerate(remaining_jobs, calibration_idx + 2):
            result = run_single_prediction_sequential(job, job_idx, len(jobs))
            if result['success']:
                completed_jobs.append(result)
            else:
                failed_jobs.append(result)
            clear_gpu_cache()
            time.sleep(1)
    elif cal_peak_vram > total_gpu_vram * vram_limit:
        # Single job already uses most of the GPU - sequential only
        print(f"\nWARNING: Calibration job used {cal_peak_vram:.1f} GB " +
              f"(>{total_gpu_vram * vram_limit:.1f} GB safe limit)")
        print(f"   Single job already fills GPU. Running remaining {len(remaining_jobs)} jobs sequentially.")

        for job_idx, job in enumerate(remaining_jobs, calibration_idx + 2):
            result = run_single_prediction_sequential(job, job_idx, len(jobs))
            if result['success']:
                completed_jobs.append(result)
            else:
                if result.get('can_retry'):
                    retried_jobs.append(job['name'])
                    clear_gpu_cache(); time.sleep(2)
                    result = run_single_prediction_sequential(job, job_idx, len(jobs), retry_with_fp32=True)
                    if result['success']:
                        completed_jobs.append(result)
                    else:
                        failed_jobs.append(result)
                else:
                    failed_jobs.append(result)
            clear_gpu_cache()
            time.sleep(1)
    else:
        # --- PHASE 2: ESTIMATE REMAINING JOBS ---
        vram_per_token = cal_peak_vram / cal_tokens
        model_overhead = cal_peak_vram * 0.3  # minimum floor

        print(f"\nPHASE 2: VRAM ESTIMATION")
        print(f"   Calibration: {cal_peak_vram:.2f} GB peak for {cal_tokens} tokens")
        print(f"   Rate: {vram_per_token:.4f} GB/token")
        print(f"   Model overhead floor: {model_overhead:.2f} GB")
        print(f"   Available for parallel: {total_gpu_vram * vram_limit:.1f} GB")

        job_estimates = []
        for job in remaining_jobs:
            tokens = count_job_tokens(job, processor)
            estimated_vram = max(
                vram_per_token * tokens * 1.15,  # 15% safety margin
                model_overhead                    # minimum floor
            )
            job_estimates.append({
                'job': job,
                'tokens': tokens,
                'estimated_vram': estimated_vram
            })
            print(f"   {job['name']}: {tokens} tokens -> ~{estimated_vram:.1f} GB estimated")

        # --- PHASE 3: PARALLEL EXECUTION ---
        print(f"\nPHASE 3: PARALLEL EXECUTION")
        safe_limit = total_gpu_vram * vram_limit
        emergency = total_gpu_vram * vram_limit

        # State tracking
        pending_estimates = list(enumerate(job_estimates, calibration_idx + 2))  # (job_number, estimate)
        running_procs = []  # list of dicts with process info
        emergency_triggered = False

        gpu_monitor.start()

        def try_launch_next():
            """Try to launch the next pending job if VRAM allows."""
            if not pending_estimates:
                return False

            current_vram = gpu_monitor.get_current()
            # Use HIGHER of actual GPU or estimated running (nvidia-smi lags behind)
            estimated_running = sum(p['estimated_vram'] for p in running_procs)
            effective_vram = max(current_vram, estimated_running)
            job_number, est = pending_estimates[0]
            needed = est['estimated_vram']

            if effective_vram + needed < safe_limit:
                pending_estimates.pop(0)
                job = est['job']
                job_name = job['name']

                # Skip existing check
                if config.get('skip_existing', False) and has_existing_output(job_name):
                    print(f"  \u23ed Skipping {job_name}: output already exists")
                    completed_jobs.append({'success': True, 'name': job_name, 'skipped': True,
                                          'structures': 0, 'time': 0, 'gpu_peak': 0,
                                          'tokens': count_job_tokens(job, processor)})
                    return True

                job_dir = get_unique_job_dir(job_name)
                os.makedirs(job_dir, exist_ok=True)

                # Generate OpenFold3 JSON query
                query_data = processor._generate_query_json(job)

                # Inject pre-computed MSA paths if available
                _job_msas_injected = False
                msa_folder_val = global_settings.get('msa_folder', '')
                if msa_folder_val:
                    precomp_msas = find_precomputed_msas(job, msa_folder_val)
                    if precomp_msas:
                        import shutil as _sh
                        id_to_name = {}
                        for seq in job.get('sequences', []):
                            if seq.get('type', '').lower() == 'protein' and seq.get('name'):
                                id_to_name[seq.get('id', '')] = seq['name']
                        for chain_entry in query_data.get('chains', []):
                            if chain_entry.get('molecule_type') != 'protein':
                                continue
                            for cid in chain_entry.get('chain_ids', []):
                                chain_name = id_to_name.get(cid)
                                if chain_name and chain_name in precomp_msas:
                                    paired_src = precomp_msas[chain_name]
                                    ind_dir = os.path.join(os.path.dirname(msa_folder_val), 'individual')
                                    ind_file = os.path.join(ind_dir, f"{chain_name}.a3m")
                                    main_src = ind_file if os.path.isfile(ind_file) else paired_src
                                    chain_msa_dir = os.path.join(job_dir, "precomputed_msas", chain_name)
                                    os.makedirs(chain_msa_dir, exist_ok=True)
                                    dropped = sanitize_a3m_for_openfold3(main_src, os.path.join(chain_msa_dir, "colabfold_main.a3m"))
                                    _sh.copy2(paired_src, os.path.join(chain_msa_dir, "colabfold_paired.a3m"))
                                    chain_entry['main_msa_file_paths'] = [chain_msa_dir]
                                    chain_entry['paired_msa_file_paths'] = [chain_msa_dir]
                                    with open(os.path.join(chain_msa_dir, "colabfold_main.a3m")) as _mf:
                                        n_seqs = sum(1 for _line in _mf if _line.startswith('>'))
                                    print(f"      {chain_name}: {n_seqs} MSA sequences" + (f" ({dropped} malformed rows removed)" if dropped else ""))
                                    break
                        _job_msas_injected = True
                full_query = {"queries": {job_name: query_data}}

                json_file = os.path.join(job_dir, f"{job_name}_query.json")
                with open(json_file, 'w') as f:
                    _json.dump(full_query, f, indent=2)

                launch_config = config.copy()
                if _job_msas_injected:
                    launch_config['_msas_injected'] = True
                cmd = build_openfold3_cmd(json_file, job_dir, launch_config)

                try:
                    proc = subprocess.Popen(
                        cmd, shell=True,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                        text=True, bufsize=1, universal_newlines=True
                    )
                except Exception as e:
                    print(f"ERROR: Failed to launch {job_name}: {e}")
                    failed_jobs.append({
                        'success': False, 'name': job_name,
                        'error': f'Launch failed: {e}', 'time': 0, 'gpu_peak': 0
                    })
                    return try_launch_next()  # try next

                # Non-blocking reader
                out_q = queue.Queue()
                reader = threading.Thread(target=_reader_thread, args=(proc.stdout, out_q), daemon=True)
                reader.start()

                info = {
                    'process': proc, 'job': job, 'job_name': job_name,
                    'job_dir': job_dir, 'json_file': json_file,
                    'job_number': job_number, 'tokens': est['tokens'],
                    'estimated_vram': needed, 'start_time': time.time(),
                    'output_queue': out_q, 'error_messages': []
                }
                running_procs.append(info)
                gpu_monitor.register_pid(proc.pid, job_name)

                print(f"\nLaunched job {job_number}/{len(jobs)}: {job_name} "
                      f"(est. {needed:.1f} GB, GPU: {effective_vram:.1f}/{safe_limit:.1f} GB)")

                time.sleep(2)  # brief pause for VRAM allocation
                return True
            return False

        # Launch initial jobs
        print(f"\nLaunching parallel jobs...")
        while try_launch_next():
            pass

        if running_procs:
            print(f"\n{len(running_procs)} job(s) running, {len(pending_estimates)} pending")
        else:
            print(f"\nWARNING: Could not launch any parallel jobs")

        # Main monitoring loop
        while running_procs or pending_estimates:
            for info in list(running_procs):
                # Drain output queue
                while not info['output_queue'].empty():
                    try:
                        line = info['output_queue'].get_nowait()
                        if not line:
                            continue
                        if 'error' in line.lower() and ('nan' in line.lower() or 'inf' in line.lower()):
                            info['error_messages'].append(line)
                        if parse_openfold3_progress(line):
                            cur = gpu_monitor.get_current()
                            print(f"   [{info['job_name']}] {line[:80]} [GPU: {cur:.1f} GB]")
                        elif ('error' in line.lower() or 'warning' in line.lower()) and not _is_suppressed_warning(line):
                            print(f"   WARNING [{info['job_name']}] {line[:80]}")
                    except queue.Empty:
                        break

                # Check completion
                rc = info['process'].poll()
                if rc is not None:
                    running_procs.remove(info)
                    job_time = time.time() - info['start_time']
                    pid_peak = gpu_monitor.unregister_pid(info['process'].pid)

                    # Final drain: capture any lines buffered between last poll and exit
                    while not info['output_queue'].empty():
                        try:
                            info['output_queue'].get_nowait()
                        except queue.Empty:
                            break

                    try:
                        if rc == 0:
                            print(f"\n{chr(0x2705)} Completed: {info['job_name']} ({job_time:.1f}s, VRAM: {pid_peak:.1f} GB)")
                            result = process_completed_job(
                                info['job_name'], info['job_dir'], info['json_file'],
                                job_time, pid_peak, drive, gdrive_folder_id
                            )
                            result['tokens'] = info['tokens']
                            if result['success']:
                                completed_jobs.append(result)
                            else:
                                failed_jobs.append(result)
                        else:
                            print(f"\n{chr(0x274c)} Failed: {info['job_name']} (exit code {rc}, {job_time:.1f}s, VRAM: {pid_peak:.1f} GB)")
                            error_files = read_error_files(info['job_dir'], info['job_name'])
                            if error_files:
                                for ef in error_files[:1]:
                                    print(f"   {ef['content'][:200]}")
                            failed_jobs.append({
                                'success': False, 'name': info['job_name'],
                                'error': f'Exit code {rc}', 'error_messages': info['error_messages'],
                                'time': job_time, 'gpu_peak': pid_peak
                            })
                    except Exception as e:
                        print(f"  ERROR processing {info['job_name']}: {e}")
                        failed_jobs.append({
                            'success': False, 'name': info['job_name'],
                            'error': str(e), 'time': job_time, 'gpu_peak': pid_peak
                        })

                    # Try launching next
                    clear_gpu_cache()
                    time.sleep(1)
                    gpu_monitor.reset_peak()

                    if not emergency_triggered:
                        while try_launch_next():
                            pass
                        if running_procs or pending_estimates:
                            print(f"   Status: {len(running_procs)} running, "
                                  f"{len(pending_estimates)} pending, "
                                  f"{len(completed_jobs)} done, {len(failed_jobs)} failed")

            # Emergency VRAM check
            current_vram = gpu_monitor.get_current()
            if current_vram > emergency and not emergency_triggered:
                emergency_triggered = True
                print(f"\nEMERGENCY: GPU VRAM at {current_vram:.1f}/{total_gpu_vram:.1f} GB")
                print(f"   Stopping new launches. Waiting for {len(running_procs)} running jobs...")

            time.sleep(2)

        peak_vram = gpu_monitor.stop()

        # If there are remaining jobs after emergency, run sequentially
        if pending_estimates:
            print(f"\nRunning {len(pending_estimates)} remaining jobs sequentially after emergency...")
            for job_number, est in pending_estimates:
                result = run_single_prediction_sequential(est['job'], job_number, len(jobs))
                if result['success']:
                    completed_jobs.append(result)
                else:
                    failed_jobs.append(result)
                clear_gpu_cache()
                time.sleep(1)

# Clean MSA cache after batch completes
shutil.rmtree('/tmp/of3_colabfold_msas', ignore_errors=True)

# ============================================================
# FINAL SUMMARY
# ============================================================
total_time = time.time() - batch_start_time

print("\n" + "=" * 60)
print("BATCH PROCESSING COMPLETE")
print("=" * 60)
print(f"Ended: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"Total duration: {total_time/60:.1f} minutes ({total_time/3600:.2f} hours)")
print(f"\nCompleted: {len(completed_jobs)}/{len(jobs)} jobs")
print(f"Failed: {len(failed_jobs)}/{len(jobs)} jobs")
if retried_jobs:
    print(f"Retried with safer settings: {len(retried_jobs)} jobs")

if completed_jobs:
    print(f"\nSuccessful jobs:")
    for job in completed_jobs:
        time_str = f"{job['time']:.1f}s" if job['time'] < 60 else f"{job['time']/60:.1f}m"
        gpu_str = f" | Peak GPU: {job['gpu_peak']:.2f} GB" if job.get('gpu_peak') else ""
        structs = job.get('structures', '?')
        print(f"  - {job['name']}: {structs} structures ({time_str}{gpu_str})")
        if job.get('url'):
            print(f"    {job['url']}")

if failed_jobs:
    print(f"\nFailed jobs:")
    for job in failed_jobs:
        error = job.get('error', 'Unknown error')[:80]
        time_str = f"{job['time']:.1f}s" if job.get('time', 0) < 60 else f"{job.get('time', 0)/60:.1f}m"
        print(f"  - {job['name']}: {error} ({time_str})")

# GPU statistics
if gpu_available and completed_jobs:
    gpu_peaks = [j['gpu_peak'] for j in completed_jobs if j.get('gpu_peak') and j['gpu_peak'] > 0]
    if gpu_peaks:
        print(f"\nGPU Peak Statistics:")
        print(f"   Average: {sum(gpu_peaks)/len(gpu_peaks):.2f} GB")
        print(f"   Highest: {max(gpu_peaks):.2f} GB")
        print(f"   Lowest: {min(gpu_peaks):.2f} GB")
        print(f"   Capacity: {total_gpu_vram:.1f} GB")
        print(f"   Peak utilization: {(max(gpu_peaks)/total_gpu_vram)*100:.1f}%")



# ============================================================
# FINAL ipSAE BATCH ANALYSIS
# ============================================================
ipsae_batch_url = None
if (ipsae_available and config.get('enable_ipsae')
        and config.get('ipsae_batch_analysis') and len(completed_jobs) > 1):
    print("\n" + "=" * 60)
    print("FINAL ipSAE BATCH ANALYSIS")
    print("=" * 60)

    ipsae_batch_output = "/content/ipsae_batch_results"
    os.makedirs(ipsae_batch_output, exist_ok=True)

    cmd_parts = [
        "ipsae-batch", "/content",
        "--backend", "openfold3",
        "--output_dir", ipsae_batch_output,
        "--pae_cutoff", str(config.get('ipsae_pae_cutoff', 10.0)),
        "--dist_cutoff", str(config.get('ipsae_dist_cutoff', 10.0)),
        "--cores", str(os.cpu_count() or 2),
    ]
    if config.get('ipsae_per_residue'):
        cmd_parts.append("--per_residue")
    if config.get('ipsae_per_contact'):
        cmd_parts.append("--per_contact")
    if config.get('ipsae_select_residues'):
        cmd_parts.extend(["--select-residues", config['ipsae_select_residues']])

    print(f"Analyzing {len(completed_jobs)} completed jobs...")
    try:
        result = subprocess.run(cmd_parts, capture_output=True, text=True, timeout=600)
        if result.returncode == 0:
            batch_files = [f for f in os.listdir(ipsae_batch_output) if not f.startswith('.')]
            print(f"Batch analysis complete: {len(batch_files)} files")
            for bf in sorted(batch_files):
                size = os.path.getsize(os.path.join(ipsae_batch_output, bf))
                print(f"   {bf} ({size//1024}KB)")

            # Upload to GDrive
            if drive and gdrive_folder_id:
                batch_zip = "/content/ipsae_batch_results.zip"
                with zipfile.ZipFile(batch_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
                    for root, dirs, files_list in os.walk(ipsae_batch_output):
                        for f in files_list:
                            fp = os.path.join(root, f)
                            arcname = os.path.join("ipsae_batch_results", os.path.relpath(fp, ipsae_batch_output))
                            zipf.write(fp, arcname)
                ipsae_batch_url = upload_to_gdrive(drive, batch_zip, gdrive_folder_id, "ipsae_batch")
                print(f"Uploaded: {ipsae_batch_url}")
            else:
                print(f"Saved locally: {ipsae_batch_output}/")
        else:
            print(f"Batch analysis failed (rc={result.returncode})")
            if result.stderr:
                print(f"   {result.stderr[:300]}")
    except subprocess.TimeoutExpired:
        print("Batch analysis timed out (600s)")
    except Exception as e:
        print(f"Batch analysis error: {e}")

# ipSAE Analysis Summary
if config.get('enable_ipsae'):
    ipsae_count = sum(1 for j in completed_jobs if j.get('ipsae_ran'))
    print(f"\nipSAE Interface Analysis:")
    print(f"   Per-job: {ipsae_count}/{len(completed_jobs)} jobs analyzed")
    if ipsae_batch_url:
        print(f"   Batch comparison: {ipsae_batch_url}")

if completed_jobs:
    avg_time = sum(j['time'] for j in completed_jobs) / len(completed_jobs)
    print(f"\nAverage time per job: {avg_time:.1f}s ({avg_time/60:.1f} min)")

if drive and gdrive_folder_id:
    print(f"\nAll results uploaded to: {gdrive_folder_name}")
    print(f"   https://drive.google.com/drive/folders/{gdrive_folder_id}")
else:
    print(f"\nAll results saved locally in job directories")

print("\nAll done!")
